# Real-World Ontology Alignment: CL + BTO Epithelial Cells

This tutorial demonstrates BOOMER's full pipeline on **real ontologies**:
the [Cell Ontology (CL)](http://obofoundry.org/ontology/cl.html) and the
[BRENDA Tissue Ontology (BTO)](http://obofoundry.org/ontology/bto.html),
seeded around **epithelial cells** (`CL:0000066`).

CL and BTO don't share cross-references directly, so we generate mapping
candidates from **shared labels** — a common real-world pattern when no
curated SSSOM file exists.

### Pipeline

1. **Download** CL and BTO OBO files
2. **Parse** each ontology into a BOOMER KB
3. **Label-match** to generate EquivalentTo candidates
4. **Merge** the two KBs with mapping candidates
5. **Extract** a module around `CL:0000066` (epithelial cell)
6. **Solve** the probabilistic alignment
7. **Export** results as SSSOM TSV and OBOGraphs JSON

## Setup

In [1]:
from pathlib import Path

from docs.tutorial.notebook_utils import show

DIR = Path("docs/tutorial/real-world-alignment-files")
DIR.mkdir(exist_ok=True)
print(f"Output directory: {DIR}")

Output directory: docs/tutorial/real-world-alignment-files


## 1. Download OBO Files

We fetch the latest CL and BTO releases from the OBO Foundry PURL service.
These are ~2-17 MB each and are `.gitignore`d.

In [2]:
%%bash
DIR=docs/tutorial/real-world-alignment-files

# Download only if not already present (avoid re-downloading on re-runs)
[ -f $DIR/cl.obo ]  || curl -sL -o $DIR/cl.obo  http://purl.obolibrary.org/obo/cl.obo
[ -f $DIR/bto.obo ] || curl -sL -o $DIR/bto.obo http://purl.obolibrary.org/obo/bto.obo

echo "Term counts:"
grep -c '^\[Term\]' $DIR/cl.obo $DIR/bto.obo

echo ""
echo "File sizes:"
ls -lh $DIR/*.obo | awk '{print $5, $NF}'

Term counts:


docs/tutorial/real-world-alignment-files/cl.obo:19026
docs/tutorial/real-world-alignment-files/bto.o

bo:6566



File sizes:


2.0M docs/tutorial/real-world-alignment-files/bto.obo
17M docs/tutorial/real-world-alignment-files/c

l.obo


## 2. Parse & Inspect

BOOMER's `obo_to_kb` converts OBO structure into hard facts (is_a, equivalent_to)
and probabilistic facts (xrefs, SKOS mappings). It also auto-generates
`MemberOfDisjointGroup` facts by ID prefix.

In [3]:
from boomer.ontology_converter import obo_to_kb

cl_kb = obo_to_kb(DIR / "cl.obo")
bto_kb = obo_to_kb(DIR / "bto.obo")

for name, kb in [("CL", cl_kb), ("BTO", bto_kb)]:
    print(f"{name}: {len(kb.facts)} hard facts, {len(kb.pfacts)} pfacts, {len(kb.labels)} labels")

print("\n--- CL labels containing 'epithelial' ---")
for eid, label in sorted(cl_kb.labels.items()):
    if "epithelial" in label.lower():
        print(f"  {eid}: {label}")

print("\n--- BTO labels containing 'epithelial' (first 15) ---")
count = 0
for eid, label in sorted(bto_kb.labels.items()):
    if "epithelial" in label.lower():
        print(f"  {eid}: {label}")
        count += 1
        if count >= 15:
            print("  ...")
            break

CL: 79363 hard facts, 32004 pfacts, 18806 labels
BTO: 11111 hard facts, 1 pfacts, 6511 labels

--- CL labels containing 'epithelial' ---
  CL:0000036: epithelial fate stem cell
  CL:0000066: epithelial cell
  CL:0000067: ciliated epithelial cell
  CL:0000068: duct epithelial cell
  CL:0000069: branched duct epithelial cell
  CL:0000072: non-branched duct epithelial cell
  CL:0000075: columnar/cuboidal epithelial cell
  CL:0000076: squamous epithelial cell
  CL:0000079: stratified epithelial cell
  CL:0000082: epithelial cell of lung
  CL:0000083: epithelial cell of pancreas
  CL:0000098: sensory epithelial cell
  CL:0000146: simple columnar epithelial cell
  CL:0000150: glandular secretory epithelial cell
  CL:0000185: myoepithelial cell
  CL:0000237: keratinizing barrier epithelial cell
  CL:0000238: non keratinizing barrier epithelial cell
  CL:0000239: brush border epithelial cell
  CL:0000240: stratified squamous epithelial cell
  CL:0000241: stratified cuboidal epithelial cell
  C

## 3. Generate Mapping Candidates from Shared Labels

Since CL and BTO don't share xrefs, we generate `EquivalentTo` candidates by
matching normalized labels. This is a simple heuristic — real pipelines would
use tools like [OAK](https://incatools.github.io/ontology-access-kit/) or
curated SSSOM files.

In [4]:
from boomer.model import EquivalentTo, PFact


def find_label_matches(kb_a, kb_b, min_length=4):
    """Find EquivalentTo candidates from shared labels."""
    b_by_label = {}
    for eid, label in kb_b.labels.items():
        norm = label.strip().lower()
        if len(norm) >= min_length:
            b_by_label.setdefault(norm, []).append(eid)

    pfacts = []
    for eid_a, label_a in kb_a.labels.items():
        for eid_b in b_by_label.get(label_a.strip().lower(), []):
            pfacts.append(
                PFact(fact=EquivalentTo(sub=eid_a, equivalent=eid_b), prob=0.7)
            )
    return pfacts


label_pfacts = find_label_matches(cl_kb, bto_kb)
print(f"Found {len(label_pfacts)} label-based mapping candidates\n")

print("Sample matches:")
for pf in label_pfacts[:10]:
    f = pf.fact
    cl_label = cl_kb.labels.get(f.sub, "?")
    bto_label = bto_kb.labels.get(f.equivalent, "?")
    print(f"  {f.sub} ({cl_label}) == {f.equivalent} ({bto_label})  p={pf.prob}")

Found 856 label-based mapping candidates

Sample matches:
  CL:0000017 (spermatocyte) == BTO:0001275 (spermatocyte)  p=0.7
  CL:0000018 (spermatid) == BTO:0001274 (spermatid)  p=0.7
  CL:0000020 (spermatogonium) == BTO:0000958 (spermatogonium)  p=0.7
  CL:0000023 (oocyte) == BTO:0000964 (oocyte)  p=0.7
  CL:0000037 (hematopoietic stem cell) == BTO:0000725 (hematopoietic stem cell)  p=0.7
  CL:0000038 (erythroid progenitor cell) == BTO:0004911 (erythroid progenitor cell)  p=0.7
  CL:0000047 (neural stem cell) == BTO:0002881 (neural stem cell)  p=0.7
  CL:0000056 (myoblast) == BTO:0000222 (myoblast)  p=0.7
  CL:0000057 (fibroblast) == BTO:0000452 (fibroblast)  p=0.7
  CL:0000058 (chondroblast) == BTO:0003607 (chondroblast)  p=0.7


We can write these candidates as a SSSOM TSV for inspection:

In [5]:
import csv
import datetime

sssom_path = DIR / "label_mappings.sssom.tsv"
all_labels = {**cl_kb.labels, **bto_kb.labels}

with open(sssom_path, "w", newline="") as fh:
    fh.write("#mapping_set_id: boomer:label_match_candidates\n")
    fh.write("#mapping_tool: boomer-label-match\n")
    fh.write(f"#mapping_date: {datetime.date.today().isoformat()}\n")
    fh.write("#curie_map:\n")
    fh.write("#  CL: http://purl.obolibrary.org/obo/CL_\n")
    fh.write("#  BTO: http://purl.obolibrary.org/obo/BTO_\n")
    fh.write("#  skos: http://www.w3.org/2004/02/skos/core#\n")
    fh.write("#  semapv: https://w3id.org/semapv/vocab/\n")

    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "subject_id", "subject_label", "predicate_id",
            "object_id", "object_label", "mapping_justification", "confidence",
        ],
        delimiter="\t",
        lineterminator="\n",
    )
    writer.writeheader()
    for pf in label_pfacts:
        f = pf.fact
        writer.writerow({
            "subject_id": f.sub,
            "subject_label": all_labels.get(f.sub, ""),
            "predicate_id": "skos:exactMatch",
            "object_id": f.equivalent,
            "object_label": all_labels.get(f.equivalent, ""),
            "mapping_justification": "semapv:LexicalMatching",
            "confidence": str(pf.prob),
        })

print(f"Wrote {len(label_pfacts)} candidates to {sssom_path}")
show(sssom_path, "tsv")

Wrote 856 candidates to docs/tutorial/real-world-alignment-files/label_mappings.sssom.tsv


**`label_mappings.sssom.tsv`**

```tsv
#mapping_set_id: boomer:label_match_candidates
#mapping_tool: boomer-label-match
#mapping_date: 2026-03-04
#curie_map:
#  CL: http://purl.obolibrary.org/obo/CL_
#  BTO: http://purl.obolibrary.org/obo/BTO_
#  skos: http://www.w3.org/2004/02/skos/core#
#  semapv: https://w3id.org/semapv/vocab/
subject_id	subject_label	predicate_id	object_id	object_label	mapping_justification	confidence
CL:0000017	spermatocyte	skos:exactMatch	BTO:0001275	spermatocyte	semapv:LexicalMatching	0.7
CL:0000018	spermatid	skos:exactMatch	BTO:0001274	spermatid	semapv:LexicalMatching	0.7
CL:0000020	spermatogonium	skos:exactMatch	BTO:0000958	spermatogonium	semapv:LexicalMatching	0.7
CL:0000023	oocyte	skos:exactMatch	BTO:0000964	oocyte	semapv:LexicalMatching	0.7
CL:0000037	hematopoietic stem cell	skos:exactMatch	BTO:0000725	hematopoietic stem cell	semapv:LexicalMatching	0.7
CL:0000038	erythroid progenitor cell	skos:exactMatch	BTO:0004911	erythroid progenitor cell	semapv:LexicalMatching	0.7
CL:0000047	neural stem cell	skos:exactMatch	BTO:0002881	neural stem cell	semapv:LexicalMatching	0.7
CL:0000056	myoblast	skos:exactMatch	BTO:0000222	myoblast	semapv:LexicalMatching	0.7
CL:0000057	fibroblast	skos:exactMatch	BTO:0000452	fibroblast	semapv:LexicalMatching	0.7
CL:0000058	chondroblast	skos:exactMatch	BTO:0003607	chondroblast	semapv:LexicalMatching	0.7
CL:0000059	ameloblast	skos:exactMatch	BTO:0001663	ameloblast	semapv:LexicalMatching	0.7
CL:0000060	odontoblast	skos:exactMatch	BTO:0001769	odontoblast	semapv:LexicalMatching	0.7
CL:0000062	osteoblast	skos:exactMatch	BTO:0001593	osteoblast	semapv:LexicalMatching	0.7
CL:0000066	epithelial cell	skos:exactMatch	BTO:0000414	epithelial cell	semapv:LexicalMatching	0.7
CL:0000091	Kupffer cell	skos:exactMatch	BTO:0000685	Kupffer cell	semapv:LexicalMatching	0.7
CL:0000092	osteoclast	skos:exactMatch	BTO:0000968	osteoclast	semapv:LexicalMatching	0.7
CL:0000094	granulocyte	skos:exactMatch	BTO:0000539	granulocyte	semapv:LexicalMatching	0.7
CL:0000097	mast cell	skos:exactMatch	BTO:0000830	mast cell	semapv:LexicalMatching	0.7
CL:0000099	interneuron	skos:exactMatch	BTO:0003811	interneuron	semapv:LexicalMatching	0.7
CL:0000108	cholinergic neuron	skos:exactMatch	BTO:0004902	cholinergic neuron	semapv:LexicalMatching	0.7
CL:0000113	mononuclear phagocyte	skos:exactMatch	BTO:0001433	mononuclear phagocyte	semapv:LexicalMatching	0.7
CL:0000115	endothelial cell	skos:exactMatch	BTO:0001176	endothelial cell	semapv:LexicalMatching	0.7
CL:0000120	granule cell	skos:exactMatch	BTO:0003393	granule cell	semapv:LexicalMatching	0.7
CL:0000125	glial cell	skos:exactMatch	BTO:0002606	glial cell	semapv:LexicalMatching	0.7
CL:0000127	astrocyte	skos:exactMatch	BTO:0000099	astrocyte	semapv:LexicalMatching	0.7
CL:0000128	oligodendrocyte	skos:exactMatch	BTO:0000962	oligodendrocyte	semapv:LexicalMatching	0.7
CL:0000134	mesenchymal stem cell	skos:exactMatch	BTO:0003298	mesenchymal stem cell	semapv:LexicalMatching	0.7
CL:0000136	adipocyte	skos:exactMatch	BTO:0000443	adipocyte	semapv:LexicalMatching	0.7
CL:0000137	osteocyte	skos:exactMatch	BTO:0002038	osteocyte	semapv:LexicalMatching	0.7
CL:0000138	chondrocyte	skos:exactMatch	BTO:0000249	chondrocyte	semapv:LexicalMatching	0.7
CL:0000142	hyalocyte	skos:exactMatch	BTO:0004271	hyalocyte	semapv:LexicalMatching	0.7
CL:0000148	melanocyte	skos:exactMatch	BTO:0000847	melanocyte	semapv:LexicalMatching	0.7
CL:0000151	secretory cell	skos:exactMatch	BTO:0003659	secretory cell	semapv:LexicalMatching	0.7
CL:0000160	goblet cell	skos:exactMatch	BTO:0001540	goblet cell	semapv:LexicalMatching	0.7
CL:0000164	enteroendocrine cell	skos:exactMatch	BTO:0003865	enteroendocrine cell	semapv:LexicalMatching	0.7
CL:0000165	neuroendocrine cell	skos:exactMatch	BTO:0002691	neuroendocrine cell	semapv:LexicalMatching	0.7
CL:0000166	chromaffin cell	skos:exactMatch	BTO:0000259	chromaffin cell	semapv:LexicalMatching	0.7
CL:0000175	luteal cell	skos:exactMatch	BTO:0003939	luteal cell	semapv:LexicalMatching	0.7
CL:0000178	Leydig cell	skos:exactMatch	BTO:0000755	Leydig cell	semapv:LexicalMatching	0.7
CL:0000182	hepatocyte	skos:exactMatch	BTO:0000575	hepatocyte	semapv:LexicalMatching	0.7
CL:0000185	myoepithelial cell	skos:exactMatch	BTO:0002309	myoepithelial cell	semapv:LexicalMatching	0.7
CL:0000192	smooth muscle cell	skos:exactMatch	BTO:0004576	smooth muscle cell	semapv:LexicalMatching	0.7
CL:0000206	chemoreceptor cell	skos:exactMatch	BTO:0005729	chemoreceptor cell	semapv:LexicalMatching	0.7
CL:0000210	photoreceptor cell	skos:exactMatch	BTO:0001060	photoreceptor cell	semapv:LexicalMatching	0.7
CL:0000214	synovial cell	skos:exactMatch	BTO:0003652	synovial cell	semapv:LexicalMatching	0.7
CL:0000216	Sertoli cell	skos:exactMatch	BTO:0001238	Sertoli cell	semapv:LexicalMatching	0.7
CL:0000218	myelinating Schwann cell	skos:exactMatch	BTO:0004996	myelinating Schwann cell	semapv:LexicalMatching	0.7
CL:0000232	erythrocyte	skos:exactMatch	BTO:0000424	erythrocyte	semapv:LexicalMatching	0.7
CL:0000234	phagocyte	skos:exactMatch	BTO:0001044	phagocyte	semapv:LexicalMatching	0.7
CL:0000235	macrophage	skos:exactMatch	BTO:0000801	macrophage	semapv:LexicalMatching	0.7
CL:0000312	keratinocyte	skos:exactMatch	BTO:0000667	keratinocyte	semapv:LexicalMatching	0.7
CL:0000317	sebocyte	skos:exactMatch	BTO:0004613	sebocyte	semapv:LexicalMatching	0.7
CL:0000346	hair follicle dermal papilla cell	skos:exactMatch	BTO:0005569	hair follicle dermal papilla cell	semapv:LexicalMatching	0.7
CL:0000362	epidermal cell	skos:exactMatch	BTO:0001470	epidermal cell	semapv:LexicalMatching	0.7
CL:0000382	scolopale cell	skos:exactMatch	BTO:0005885	scolopale cell	semapv:LexicalMatching	0.7
CL:0000421	coelomocyte	skos:exactMatch	BTO:0002856	coelomocyte	semapv:LexicalMatching	0.7
CL:0000442	follicular dendritic cell	skos:exactMatch	BTO:0004267	follicular dendritic cell	semapv:LexicalMatching	0.7
CL:0000448	white adipocyte	skos:exactMatch	BTO:0006429	white adipocyte	semapv:LexicalMatching	0.7
CL:0000449	brown adipocyte	skos:exactMatch	BTO:0006430	brown adipocyte	semapv:LexicalMatching	0.7
CL:0000451	dendritic cell	skos:exactMatch	BTO:0002042	dendritic cell	semapv:LexicalMatching	0.7
CL:0000453	Langerhans cell	skos:exactMatch	BTO:0000705	Langerhans cell	semapv:LexicalMatching	0.7
CL:0000486	garland cell	skos:exactMatch	BTO:0004596	garland cell	semapv:LexicalMatching	0.7
CL:0000499	stromal cell	skos:exactMatch	BTO:0002064	stromal cell	semapv:LexicalMatching	0.7
CL:0000501	granulosa cell	skos:exactMatch	BTO:0000542	granulosa cell	semapv:LexicalMatching	0.7
CL:0000503	theca cell	skos:exactMatch	BTO:0002850	theca cell	semapv:LexicalMatching	0.7
CL:0000504	enterochromaffin-like cell	skos:exactMatch	BTO:0002692	enterochromaffin-like cell	semapv:LexicalMatching	0.7
CL:0000510	paneth cell	skos:exactMatch	BTO:0000993	paneth cell	semapv:LexicalMatching	0.7
CL:0000540	neuron	skos:exactMatch	BTO:0000938	neuron	semapv:LexicalMatching	0.7
CL:0000541	melanoblast	skos:exactMatch	BTO:0003217	melanoblast	semapv:LexicalMatching	0.7
CL:0000542	lymphocyte	skos:exactMatch	BTO:0000775	lymphocyte	semapv:LexicalMatching	0.7
CL:0000556	megakaryocyte	skos:exactMatch	BTO:0000843	megakaryocyte	semapv:LexicalMatching	0.7
CL:0000558	reticulocyte	skos:exactMatch	BTO:0001173	reticulocyte	semapv:LexicalMatching	0.7
CL:0000561	amacrine cell	skos:exactMatch	BTO:0004044	amacrine cell	semapv:LexicalMatching	0.7
CL:0000563	endospore	skos:exactMatch	BTO:0002779	endospore	semapv:LexicalMatching	0.7
CL:0000575	corneal epithelial cell	skos:exactMatch	BTO:0004298	corneal epithelial cell	semapv:LexicalMatching	0.7
CL:0000576	monocyte	skos:exactMatch	BTO:0000876	monocyte	semapv:LexicalMatching	0.7
CL:0000580	neutrophilic myelocyte	skos:exactMatch	BTO:0003455	neutrophilic myelocyte	semapv:LexicalMatching	0.7
CL:0000581	peritoneal macrophage	skos:exactMatch	BTO:0001034	peritoneal macrophage	semapv:LexicalMatching	0.7
CL:0000583	alveolar macrophage	skos:exactMatch	BTO:0000802	alveolar macrophage	semapv:LexicalMatching	0.7
CL:0000584	enterocyte	skos:exactMatch	BTO:0000398	enterocyte	semapv:LexicalMatching	0.7
CL:0000586	germ cell	skos:exactMatch	BTO:0000535	germ cell	semapv:LexicalMatching	0.7
CL:0000588	odontoclast	skos:exactMatch	BTO:0002516	odontoclast	semapv:LexicalMatching	0.7
CL:0000590	small luteal cell	skos:exactMatch	BTO:0005679	small luteal cell	semapv:LexicalMatching	0.7
CL:0000592	large luteal cell	skos:exactMatch	BTO:0005680	large luteal cell	semapv:LexicalMatching	0.7
CL:0000594	skeletal muscle satellite cell	skos:exactMatch	BTO:0005787	skeletal muscle satellite cell	semapv:LexicalMatching	0.7
CL:0000598	pyramidal neuron	skos:exactMatch	BTO:0003102	pyramidal neuron	semapv:LexicalMatching	0.7
CL:0000599	conidium	skos:exactMatch	BTO:0000283	conidium	semapv:LexicalMatching	0.7
CL:0000607	ascospore	skos:exactMatch	BTO:0000097	ascospore	semapv:LexicalMatching	0.7
CL:0000609	vestibular hair cell	skos:exactMatch	BTO:0005824	vestibular hair cell	semapv:LexicalMatching	0.7
CL:0000612	eosinophilic myelocyte	skos:exactMatch	BTO:0003454	eosinophilic myelocyte	semapv:LexicalMatching	0.7
CL:0000614	basophilic myelocyte	skos:exactMatch	BTO:0003456	basophilic myelocyte	semapv:LexicalMatching	0.7
CL:0000615	basidiospore	skos:exactMatch	BTO:0002166	basidiospore	semapv:LexicalMatching	0.7
CL:0000623	natural killer cell	skos:exactMatch	BTO:0000914	natural killer cell	semapv:LexicalMatching	0.7
CL:0000630	supporting cell	skos:exactMatch	BTO:0002315	supporting cell	semapv:LexicalMatching	0.7
CL:0000632	hepatic stellate cell	skos:exactMatch	BTO:0002741	hepatic stellate cell	semapv:LexicalMatching	0.7
CL:0000636	Mueller cell	skos:exactMatch	BTO:0003064	Mueller cell	semapv:LexicalMatching	0.7
CL:0000645	pituicyte	skos:exactMatch	BTO:0003793	pituicyte	semapv:LexicalMatching	0.7
CL:0000646	basal cell	skos:exactMatch	BTO:0000939	basal cell	semapv:LexicalMatching	0.7
CL:0000650	mesangial cell	skos:exactMatch	BTO:0000853	mesangial cell	semapv:LexicalMatching	0.7
CL:0000652	pinealocyte	skos:exactMatch	BTO:0001068	pinealocyte	semapv:LexicalMatching	0.7
CL:0000653	podocyte	skos:exactMatch	BTO:0002295	podocyte	semapv:LexicalMatching	0.7
CL:0000654	primary oocyte	skos:exactMatch	BTO:0000512	primary oocyte	semapv:LexicalMatching	0.7
CL:0000655	secondary oocyte	skos:exactMatch	BTO:0003094	secondary oocyte	semapv:LexicalMatching	0.7
CL:0000656	primary spermatocyte	skos:exactMatch	BTO:0001115	primary spermatocyte	semapv:LexicalMatching	0.7
CL:0000657	secondary spermatocyte	skos:exactMatch	BTO:0000709	secondary spermatocyte	semapv:LexicalMatching	0.7
CL:0000669	pericyte	skos:exactMatch	BTO:0002441	pericyte	semapv:LexicalMatching	0.7
CL:0000673	Kenyon cell	skos:exactMatch	BTO:0006033	Kenyon cell	semapv:LexicalMatching	0.7
CL:0000700	dopaminergic neuron	skos:exactMatch	BTO:0004032	dopaminergic neuron	semapv:LexicalMatching	0.7
CL:0000724	heterocyst	skos:exactMatch	BTO:0000600	heterocyst	semapv:LexicalMatching	0.7
CL:0000737	striated muscle cell	skos:exactMatch	BTO:0002916	striated muscle cell	semapv:LexicalMatching	0.7
CL:0000738	leukocyte	skos:exactMatch	BTO:0000751	leukocyte	semapv:LexicalMatching	0.7
CL:0000740	retinal ganglion cell	skos:exactMatch	BTO:0001800	retinal ganglion cell	semapv:LexicalMatching	0.7
CL:0000765	erythroblast	skos:exactMatch	BTO:0001571	erythroblast	semapv:LexicalMatching	0.7
CL:0000767	basophil	skos:exactMatch	BTO:0000129	basophil	semapv:LexicalMatching	0.7
CL:0000771	eosinophil	skos:exactMatch	BTO:0000399	eosinophil	semapv:LexicalMatching	0.7
CL:0000775	neutrophil	skos:exactMatch	BTO:0000130	neutrophil	semapv:LexicalMatching	0.7
CL:0000782	myeloid dendritic cell	skos:exactMatch	BTO:0004721	myeloid dendritic cell	semapv:LexicalMatching	0.7
CL:0000784	plasmacytoid dendritic cell	skos:exactMatch	BTO:0004625	plasmacytoid dendritic cell	semapv:LexicalMatching	0.7
CL:0000786	plasma cell	skos:exactMatch	BTO:0000392	plasma cell	semapv:LexicalMatching	0.7
CL:0000835	myeloblast	skos:exactMatch	BTO:0000187	myeloblast	semapv:LexicalMatching	0.7
CL:0000836	promyelocyte	skos:exactMatch	BTO:0005790	promyelocyte	semapv:LexicalMatching	0.7
CL:0000850	serotonergic neuron	skos:exactMatch	BTO:0006205	serotonergic neuron	semapv:LexicalMatching	0.7
CL:0000863	M1 macrophage	skos:exactMatch	BTO:0006110	M1 macrophage	semapv:LexicalMatching	0.7
CL:0000890	M2 macrophage	skos:exactMatch	BTO:0006111	M2 macrophage	semapv:LexicalMatching	0.7
CL:0000891	foam cell	skos:exactMatch	BTO:0003872	foam cell	semapv:LexicalMatching	0.7
CL:0000893	thymocyte	skos:exactMatch	BTO:0001372	thymocyte	semapv:LexicalMatching	0.7
CL:0000917	Tc1 cell	skos:exactMatch	BTO:0004793	TC1 cell	semapv:LexicalMatching	0.7
CL:0000918	Tc2 cell	skos:exactMatch	BTO:0004794	TC2 cell	semapv:LexicalMatching	0.7
CL:0000988	hematopoietic cell	skos:exactMatch	BTO:0000574	hematopoietic cell	semapv:LexicalMatching	0.7
CL:0001006	dermal dendritic cell	skos:exactMatch	BTO:0004812	dermal dendritic cell	semapv:LexicalMatching	0.7
CL:0001031	cerebellar granule cell	skos:exactMatch	BTO:0004278	cerebellar granule cell	semapv:LexicalMatching	0.7
CL:0002064	pancreatic acinar cell	skos:exactMatch	BTO:0000028	pancreatic acinar cell	semapv:LexicalMatching	0.7
CL:0002085	tanycyte	skos:exactMatch	BTO:0001953	tanycyte	semapv:LexicalMatching	0.7
CL:0002088	interstitial cell of Cajal	skos:exactMatch	BTO:0003914	interstitial cell of Cajal	semapv:LexicalMatching	0.7
CL:0002092	bone marrow cell	skos:exactMatch	BTO:0004850	bone marrow cell	semapv:LexicalMatching	0.7
CL:0002144	capillary endothelial cell	skos:exactMatch	BTO:0004956	capillary endothelial cell	semapv:LexicalMatching	0.7
CL:0002153	corneocyte	skos:exactMatch	BTO:0001943	corneocyte	semapv:LexicalMatching	0.7
CL:0002173	extraglomerular mesangial cell	skos:exactMatch	BTO:0005159	extraglomerular mesangial cell	semapv:LexicalMatching	0.7
CL:0002188	glomerular endothelial cell	skos:exactMatch	BTO:0004632	glomerular endothelial cell	semapv:LexicalMatching	0.7
CL:0002193	myelocyte	skos:exactMatch	BTO:0000734	myelocyte	semapv:LexicalMatching	0.7
CL:0002246	peripheral blood stem cell	skos:exactMatch	BTO:0002669	peripheral blood stem cell	semapv:LexicalMatching	0.7
CL:0002248	pluripotent stem cell	skos:exactMatch	BTO:0006078	pluripotent stem cell	semapv:LexicalMatching	0.7
CL:0002275	pancreatic PP cell	skos:exactMatch	BTO:0000805	pancreatic PP cell	semapv:LexicalMatching	0.7
CL:0002322	embryonic stem cell	skos:exactMatch	BTO:0001086	embryonic stem cell	semapv:LexicalMatching	0.7
CL:0002323	amniocyte	skos:exactMatch	BTO:0000066	amniocyte	semapv:LexicalMatching	0.7
CL:0002328	bronchial epithelial cell	skos:exactMatch	BTO:0002922	bronchial epithelial cell	semapv:LexicalMatching	0.7
CL:0002334	preadipocyte	skos:exactMatch	BTO:0001107	preadipocyte	semapv:LexicalMatching	0.7
CL:0002363	keratocyte	skos:exactMatch	BTO:0005170	keratocyte	semapv:LexicalMatching	0.7
CL:0002366	myometrial cell	skos:exactMatch	BTO:0004519	myometrial cell	semapv:LexicalMatching	0.7
CL:0002372	myotube	skos:exactMatch	BTO:0001760	myotube	semapv:LexicalMatching	0.7
CL:0002376	non-myelinating Schwann cell	skos:exactMatch	BTO:0004997	non-myelinating Schwann cell	semapv:LexicalMatching	0.7
CL:0002488	trophoblast giant cell	skos:exactMatch	BTO:0005362	trophoblast giant cell	semapv:LexicalMatching	0.7
CL:0002520	nephrocyte	skos:exactMatch	BTO:0004597	nephrocyte	semapv:LexicalMatching	0.7
CL:0002539	aortic smooth muscle cell	skos:exactMatch	BTO:0004577	aortic smooth muscle cell	semapv:LexicalMatching	0.7
CL:0002544	aortic endothelial cell	skos:exactMatch	BTO:0003245	aortic endothelial cell	semapv:LexicalMatching	0.7
CL:0002563	intestinal epithelial cell	skos:exactMatch	BTO:0005937	intestinal epithelial cell	semapv:LexicalMatching	0.7
CL:0002573	Schwann cell	skos:exactMatch	BTO:0001220	Schwann cell	semapv:LexicalMatching	0.7
CL:0002598	bronchial smooth muscle cell	skos:exactMatch	BTO:0004402	bronchial smooth muscle cell	semapv:LexicalMatching	0.7
CL:0002620	skin fibroblast	skos:exactMatch	BTO:0001255	skin fibroblast	semapv:LexicalMatching	0.7
CL:0007010	preosteoblast	skos:exactMatch	BTO:0002051	preosteoblast	semapv:LexicalMatching	0.7
CL:0007011	enteric neuron	skos:exactMatch	BTO:0006389	enteric neuron	semapv:LexicalMatching	0.7
CL:0008002	skeletal muscle fiber	skos:exactMatch	BTO:0002319	skeletal muscle fiber	semapv:LexicalMatching	0.7
CL:0008011	skeletal muscle satellite stem cell	skos:exactMatch	BTO:0006105	skeletal muscle satellite stem cell	semapv:LexicalMatching	0.7
CL:0008012	quiescent skeletal muscle satellite cell	skos:exactMatch	BTO:0006103	quiescent skeletal muscle satellite cell	semapv:LexicalMatching	0.7
CL:0008016	activated skeletal muscle satellite cell	skos:exactMatch	BTO:0006102	activated skeletal muscle satellite cell	semapv:LexicalMatching	0.7
CL:0008019	mesenchymal cell	skos:exactMatch	BTO:0002625	mesenchymal cell	semapv:LexicalMatching	0.7
CL:0008034	mural cell	skos:exactMatch	BTO:0004720	mural cell	semapv:LexicalMatching	0.7
CL:0008036	extravillous trophoblast	skos:exactMatch	BTO:0002366	extravillous trophoblast	semapv:LexicalMatching	0.7
CL:0009002	inflammatory cell	skos:exactMatch	BTO:0003861	inflammatory cell	semapv:LexicalMatching	0.7
CL:0010017	zygote	skos:exactMatch	BTO:0000854	zygote	semapv:LexicalMatching	0.7
CL:0011012	neural crest cell	skos:exactMatch	BTO:0003825	neural crest cell	semapv:LexicalMatching	0.7
CL:0011028	olfactory ensheathing cell	skos:exactMatch	BTO:0002771	olfactory ensheathing cell	semapv:LexicalMatching	0.7
CL:0011030	dermal microvascular endothelial cell	skos:exactMatch	BTO:0004574	dermal microvascular endothelial cell	semapv:LexicalMatching	0.7
CL:0011031	monocyte-derived dendritic cell	skos:exactMatch	BTO:0002900	monocyte-derived dendritic cell	semapv:LexicalMatching	0.7
CL:0011110	histaminergic neuron	skos:exactMatch	BTO:0005819	histaminergic neuron	semapv:LexicalMatching	0.7
CL:0017005	lymphoblast	skos:exactMatch	BTO:0000772	lymphoblast	semapv:LexicalMatching	0.7
CL:0017006	B-lymphoblast	skos:exactMatch	BTO:0001528	B-lymphoblast	semapv:LexicalMatching	0.7
CL:1000347	colonocyte	skos:exactMatch	BTO:0000275	colonocyte	semapv:LexicalMatching	0.7
CL:1000488	cholangiocyte	skos:exactMatch	BTO:0003164	cholangiocyte	semapv:LexicalMatching	0.7
CL:1001435	periglomerular cell	skos:exactMatch	BTO:0003796	periglomerular cell	semapv:LexicalMatching	0.7
CL:1001474	medium spiny neuron	skos:exactMatch	BTO:0004778	medium spiny neuron	semapv:LexicalMatching	0.7
CL:1001502	mitral cell	skos:exactMatch	BTO:0001283	mitral cell	semapv:LexicalMatching	0.7
CL:1001567	lung endothelial cell	skos:exactMatch	BTO:0005812	lung endothelial cell	semapv:LexicalMatching	0.7
CL:1001568	pulmonary artery endothelial cell	skos:exactMatch	BTO:0001141	pulmonary artery endothelial cell	semapv:LexicalMatching	0.7
CL:2000001	peripheral blood mononuclear cell	skos:exactMatch	BTO:0001025	peripheral blood mononuclear cell	semapv:LexicalMatching	0.7
CL:2000002	decidual cell	skos:exactMatch	BTO:0002770	decidual cell	semapv:LexicalMatching	0.7
CL:2000008	microvascular endothelial cell	skos:exactMatch	BTO:0003123	microvascular endothelial cell	semapv:LexicalMatching	0.7
CL:2000016	lung microvascular endothelial cell	skos:exactMatch	BTO:0005738	lung microvascular endothelial cell	semapv:LexicalMatching	0.7
CL:2000042	embryonic fibroblast	skos:exactMatch	BTO:0004725	embryonic fibroblast	semapv:LexicalMatching	0.7
CL:2000044	brain microvascular endothelial cell	skos:exactMatch	BTO:0003636	brain microvascular endothelial cell	semapv:LexicalMatching	0.7
CL:2000052	umbilical artery endothelial cell	skos:exactMatch	BTO:0004295	umbilical artery endothelial cell	semapv:LexicalMatching	0.7
CL:2000064	ovarian surface epithelial cell	skos:exactMatch	BTO:0004484	ovarian surface epithelial cell	semapv:LexicalMatching	0.7
CL:2000074	splenocyte	skos:exactMatch	BTO:0001598	splenocyte	semapv:LexicalMatching	0.7
CL:4030030	peripheral blood lymphocyte	skos:exactMatch	BTO:0004729	peripheral blood lymphocyte	semapv:LexicalMatching	0.7
CL:4030031	interstitial cell	skos:exactMatch	BTO:0004238	interstitial cell	semapv:LexicalMatching	0.7
CL:4033014	peg cell	skos:exactMatch	BTO:0005588	peg cell	semapv:LexicalMatching	0.7
CL:4033050	catecholaminergic neuron	skos:exactMatch	BTO:0004033	catecholaminergic neuron	semapv:LexicalMatching	0.7
DHBA:10153	neural plate	skos:exactMatch	BTO:0001765	neural plate	semapv:LexicalMatching	0.7
DHBA:10154	neural tube	skos:exactMatch	BTO:0001057	neural tube	semapv:LexicalMatching	0.7
DHBA:10155	brain	skos:exactMatch	BTO:0000142	brain	semapv:LexicalMatching	0.7
DHBA:10158	telencephalon	skos:exactMatch	BTO:0000239	telencephalon	semapv:LexicalMatching	0.7
DHBA:10159	cerebral cortex	skos:exactMatch	BTO:0000233	cerebral cortex	semapv:LexicalMatching	0.7
DHBA:10302	subiculum	skos:exactMatch	BTO:0003087	subiculum	semapv:LexicalMatching	0.7
DHBA:10307	olfactory bulb	skos:exactMatch	BTO:0000961	olfactory bulb	semapv:LexicalMatching	0.7
DHBA:10308	anterior olfactory nucleus	skos:exactMatch	BTO:0003649	anterior olfactory nucleus	semapv:LexicalMatching	0.7
DHBA:10310	olfactory tubercle	skos:exactMatch	BTO:0001869	olfactory tubercle	semapv:LexicalMatching	0.7
DHBA:10334	caudate nucleus	skos:exactMatch	BTO:0000211	caudate nucleus	semapv:LexicalMatching	0.7
DHBA:10339	nucleus accumbens	skos:exactMatch	BTO:0001862	nucleus accumbens	semapv:LexicalMatching	0.7
DHBA:10342	globus pallidus	skos:exactMatch	BTO:0002246	globus pallidus	semapv:LexicalMatching	0.7
DHBA:10349	basal forebrain	skos:exactMatch	BTO:0002444	basal forebrain	semapv:LexicalMatching	0.7
DHBA:10389	diencephalon	skos:exactMatch	BTO:0000342	diencephalon	semapv:LexicalMatching	0.7
DHBA:10390	thalamus	skos:exactMatch	BTO:0001365	thalamus	semapv:LexicalMatching	0.7
DHBA:10428	posterior nuclear complex of thalamus	skos:exactMatch	BTO:0002462	posterior nuclear complex of thalamus	semapv:LexicalMatching	0.7
DHBA:10429	lateral geniculate nucleus	skos:exactMatch	BTO:0004366	lateral geniculate nucleus	semapv:LexicalMatching	0.7
DHBA:10451	epithalamus	skos:exactMatch	BTO:0000175	epithalamus	semapv:LexicalMatching	0.7
DHBA:10463	zona incerta	skos:exactMatch	BTO:0003146	zona incerta	semapv:LexicalMatching	0.7
DHBA:10466	subthalamic nucleus	skos:exactMatch	BTO:0002252	subthalamic nucleus	semapv:LexicalMatching	0.7
DHBA:10467	hypothalamus	skos:exactMatch	BTO:0000614	hypothalamus	semapv:LexicalMatching	0.7
DHBA:10476	paraventricular nucleus of hypothalamus	skos:exactMatch	BTO:0002476	paraventricular nucleus of hypothalamus	semapv:LexicalMatching	0.7
DHBA:10480	suprachiasmatic nucleus	skos:exactMatch	BTO:0001822	suprachiasmatic nucleus	semapv:LexicalMatching	0.7
DHBA:10542	ventricular zone	skos:exactMatch	BTO:0003654	ventricular zone	semapv:LexicalMatching	0.7
DHBA:10561	corpus callosum	skos:exactMatch	BTO:0000615	corpus callosum	semapv:LexicalMatching	0.7
DHBA:10655	metencephalon	skos:exactMatch	BTO:0000673	metencephalon	semapv:LexicalMatching	0.7
DHBA:10656	cerebellum	skos:exactMatch	BTO:0000232	cerebellum	semapv:LexicalMatching	0.7
DHBA:10657	cerebellar cortex	skos:exactMatch	BTO:0000043	cerebellar cortex	semapv:LexicalMatching	0.7
DHBA:10661	pons	skos:exactMatch	BTO:0001101	pons	semapv:LexicalMatching	0.7
DHBA:12073	olfactory tract	skos:exactMatch	BTO:0003647	olfactory tract	semapv:LexicalMatching	0.7
DHBA:12101	subcommissural organ	skos:exactMatch	BTO:0001820	subcommissural organ	semapv:LexicalMatching	0.7
DHBA:12104	subfornical organ	skos:exactMatch	BTO:0006266	subfornical organ	semapv:LexicalMatching	0.7
DHBA:12106	organum vasculosum laminae terminalis	skos:exactMatch	BTO:0006268	organum vasculosum laminae terminalis	semapv:LexicalMatching	0.7
DHBA:12113	frontal lobe	skos:exactMatch	BTO:0000484	frontal lobe	semapv:LexicalMatching	0.7
DHBA:12131	parietal lobe	skos:exactMatch	BTO:0001001	parietal lobe	semapv:LexicalMatching	0.7
DHBA:12139	temporal lobe	skos:exactMatch	BTO:0001355	temporal lobe	semapv:LexicalMatching	0.7
DHBA:12198	oculomotor nucleus	skos:exactMatch	BTO:0006504	oculomotor nucleus	semapv:LexicalMatching	0.7
DHBA:12251	substantia nigra	skos:exactMatch	BTO:0000143	substantia nigra	semapv:LexicalMatching	0.7
DHBA:12406	pontine nucleus	skos:exactMatch	BTO:0003396	pontine nucleus	semapv:LexicalMatching	0.7
DHBA:12776	corticospinal tract	skos:exactMatch	BTO:0006132	corticospinal tract	semapv:LexicalMatching	0.7
DHBA:12782	rubrospinal tract	skos:exactMatch	BTO:0006159	rubrospinal tract	semapv:LexicalMatching	0.7
DHBA:12805	fourth ventricle	skos:exactMatch	BTO:0003426	fourth ventricle	semapv:LexicalMatching	0.7
DHBA:12807	area postrema	skos:exactMatch	BTO:0003425	area postrema	semapv:LexicalMatching	0.7
DHBA:12890	spinal cord	skos:exactMatch	BTO:0001279	spinal cord	semapv:LexicalMatching	0.7
DHBA:13338	median eminence	skos:exactMatch	BTO:0001954	median eminence	semapv:LexicalMatching	0.7
DHBA:15544	optic nerve	skos:exactMatch	BTO:0000966	optic nerve	semapv:LexicalMatching	0.7
GO:0007588	excretion	skos:exactMatch	BTO:0000491	excretion	semapv:LexicalMatching	0.7
GO:0031143	pseudopodium	skos:exactMatch	BTO:0002610	pseudopodium	semapv:LexicalMatching	0.7
GO:0042151	nematocyst	skos:exactMatch	BTO:0000918	nematocyst	semapv:LexicalMatching	0.7
GO:0043209	myelin sheath	skos:exactMatch	BTO:0000900	myelin sheath	semapv:LexicalMatching	0.7
GO:0046903	secretion	skos:exactMatch	BTO:0002979	secretion	semapv:LexicalMatching	0.7
GO:0097457	hippocampal mossy fiber	skos:exactMatch	BTO:0002654	hippocampal mossy fiber	semapv:LexicalMatching	0.7
MBA:1065	Hindbrain	skos:exactMatch	BTO:0000672	hindbrain	semapv:LexicalMatching	0.7
MBA:10671	Median eminence	skos:exactMatch	BTO:0001954	median eminence	semapv:LexicalMatching	0.7
MBA:108	choroid plexus	skos:exactMatch	BTO:0000258	choroid plexus	semapv:LexicalMatching	0.7
MBA:1097	Hypothalamus	skos:exactMatch	BTO:0000614	hypothalamus	semapv:LexicalMatching	0.7
MBA:145	fourth ventricle	skos:exactMatch	BTO:0003426	fourth ventricle	semapv:LexicalMatching	0.7
MBA:147	Locus ceruleus	skos:exactMatch	BTO:0001408	locus ceruleus	semapv:LexicalMatching	0.7
MBA:159	Anterior olfactory nucleus	skos:exactMatch	BTO:0003649	anterior olfactory nucleus	semapv:LexicalMatching	0.7
MBA:207	Area postrema	skos:exactMatch	BTO:0003425	area postrema	semapv:LexicalMatching	0.7
MBA:286	Suprachiasmatic nucleus	skos:exactMatch	BTO:0001822	suprachiasmatic nucleus	semapv:LexicalMatching	0.7
MBA:304325711	retina	skos:exactMatch	BTO:0001175	retina	semapv:LexicalMatching	0.7
MBA:313	Midbrain	skos:exactMatch	BTO:0000138	midbrain	semapv:LexicalMatching	0.7
MBA:338	Subfornical organ	skos:exactMatch	BTO:0006266	subfornical organ	semapv:LexicalMatching	0.7
MBA:343	Brain stem	skos:exactMatch	BTO:0000146	brain stem	semapv:LexicalMatching	0.7
MBA:35	Oculomotor nucleus	skos:exactMatch	BTO:0006504	oculomotor nucleus	semapv:LexicalMatching	0.7
MBA:470	Subthalamic nucleus	skos:exactMatch	BTO:0002252	subthalamic nucleus	semapv:LexicalMatching	0.7
MBA:502	Subiculum	skos:exactMatch	BTO:0003087	subiculum	semapv:LexicalMatching	0.7
MBA:512	Cerebellum	skos:exactMatch	BTO:0000232	cerebellum	semapv:LexicalMatching	0.7
MBA:528	Cerebellar cortex	skos:exactMatch	BTO:0000043	cerebellar cortex	semapv:LexicalMatching	0.7
MBA:549	Thalamus	skos:exactMatch	BTO:0001365	thalamus	semapv:LexicalMatching	0.7
MBA:56	Nucleus accumbens	skos:exactMatch	BTO:0001862	nucleus accumbens	semapv:LexicalMatching	0.7
MBA:599626923	Subcommissural organ	skos:exactMatch	BTO:0001820	subcommissural organ	semapv:LexicalMatching	0.7
MBA:688	Cerebral cortex	skos:exactMatch	BTO:0000233	cerebral cortex	semapv:LexicalMatching	0.7
MBA:726	Dentate gyrus	skos:exactMatch	BTO:0002496	dentate gyrus	semapv:LexicalMatching	0.7
MBA:754	Olfactory tubercle	skos:exactMatch	BTO:0001869	olfactory tubercle	semapv:LexicalMatching	0.7
MBA:771	Pons	skos:exactMatch	BTO:0001101	pons	semapv:LexicalMatching	0.7
MBA:776	corpus callosum	skos:exactMatch	BTO:0000615	corpus callosum	semapv:LexicalMatching	0.7
MBA:784	corticospinal tract	skos:exactMatch	BTO:0006132	corticospinal tract	semapv:LexicalMatching	0.7
MBA:797	Zona incerta	skos:exactMatch	BTO:0003146	zona incerta	semapv:LexicalMatching	0.7
MBA:808	glossopharyngeal nerve	skos:exactMatch	BTO:0004979	glossopharyngeal nerve	semapv:LexicalMatching	0.7
MBA:81	lateral ventricle	skos:exactMatch	BTO:0000879	lateral ventricle	semapv:LexicalMatching	0.7
MBA:813	hypoglossal nerve	skos:exactMatch	BTO:0003386	hypoglossal nerve	semapv:LexicalMatching	0.7
MBA:840	olfactory nerve	skos:exactMatch	BTO:0003648	olfactory nerve	semapv:LexicalMatching	0.7
MBA:848	optic nerve	skos:exactMatch	BTO:0000966	optic nerve	semapv:LexicalMatching	0.7
MBA:863	rubrospinal tract	skos:exactMatch	BTO:0006159	rubrospinal tract	semapv:LexicalMatching	0.7
MBA:901	trigeminal nerve	skos:exactMatch	BTO:0001072	trigeminal nerve	semapv:LexicalMatching	0.7
MBA:917	vagus nerve	skos:exactMatch	BTO:0003472	vagus nerve	semapv:LexicalMatching	0.7
MBA:933	vestibulocochlear nerve	skos:exactMatch	BTO:0003490	vestibulocochlear nerve	semapv:LexicalMatching	0.7
MBA:958	Epithalamus	skos:exactMatch	BTO:0000175	epithalamus	semapv:LexicalMatching	0.7
MBA:997	brain	skos:exactMatch	BTO:0000142	brain	semapv:LexicalMatching	0.7
NCBITaxon:1	root	skos:exactMatch	BTO:0001188	root	semapv:LexicalMatching	0.7
PATO:0001190	juvenile	skos:exactMatch	BTO:0002168	juvenile	semapv:LexicalMatching	0.7
UBERON:0000002	uterine cervix	skos:exactMatch	BTO:0001421	uterine cervix	semapv:LexicalMatching	0.7
UBERON:0000004	nose	skos:exactMatch	BTO:0000840	nose	semapv:LexicalMatching	0.7
UBERON:0000009	submucosa	skos:exactMatch	BTO:0002107	submucosa	semapv:LexicalMatching	0.7
UBERON:0000010	peripheral nervous system	skos:exactMatch	BTO:0001028	peripheral nervous system	semapv:LexicalMatching	0.7
UBERON:0000011	parasympathetic nervous system	skos:exactMatch	BTO:0001833	parasympathetic nervous system	semapv:LexicalMatching	0.7
UBERON:0000013	sympathetic nervous system	skos:exactMatch	BTO:0001832	sympathetic nervous system	semapv:LexicalMatching	0.7
UBERON:0000016	endocrine pancreas	skos:exactMatch	BTO:0000650	endocrine pancreas	semapv:LexicalMatching	0.7
UBERON:0000017	exocrine pancreas	skos:exactMatch	BTO:0000434	exocrine pancreas	semapv:LexicalMatching	0.7
UBERON:0000018	compound eye	skos:exactMatch	BTO:0001921	compound eye	semapv:LexicalMatching	0.7
UBERON:0000020	sense organ	skos:exactMatch	BTO:0000202	sense organ	semapv:LexicalMatching	0.7
UBERON:0000029	lymph node	skos:exactMatch	BTO:0000784	lymph node	semapv:LexicalMatching	0.7
UBERON:0000030	lamina propria	skos:exactMatch	BTO:0002330	lamina propria	semapv:LexicalMatching	0.7
UBERON:0000033	head	skos:exactMatch	BTO:0000282	head	semapv:LexicalMatching	0.7
UBERON:0000038	follicular fluid	skos:exactMatch	BTO:0004383	follicular fluid	semapv:LexicalMatching	0.7
UBERON:0000043	tendon	skos:exactMatch	BTO:0001356	tendon	semapv:LexicalMatching	0.7
UBERON:0000045	ganglion	skos:exactMatch	BTO:0000497	ganglion	semapv:LexicalMatching	0.7
UBERON:0000053	macula lutea	skos:exactMatch	BTO:0003015	macula lutea	semapv:LexicalMatching	0.7
UBERON:0000056	ureter	skos:exactMatch	BTO:0001409	ureter	semapv:LexicalMatching	0.7
UBERON:0000057	urethra	skos:exactMatch	BTO:0001426	urethra	semapv:LexicalMatching	0.7
UBERON:0000059	large intestine	skos:exactMatch	BTO:0000706	large intestine	semapv:LexicalMatching	0.7
UBERON:0000074	renal glomerulus	skos:exactMatch	BTO:0000530	renal glomerulus	semapv:LexicalMatching	0.7
UBERON:0000076	external ectoderm	skos:exactMatch	BTO:0006242	external ectoderm	semapv:LexicalMatching	0.7
UBERON:0000079	male reproductive system	skos:exactMatch	BTO:0000082	male reproductive system	semapv:LexicalMatching	0.7
UBERON:0000084	ureteric bud	skos:exactMatch	BTO:0001646	ureteric bud	semapv:LexicalMatching	0.7
UBERON:0000085	morula	skos:exactMatch	BTO:0001508	morula	semapv:LexicalMatching	0.7
UBERON:0000086	zona pellucida	skos:exactMatch	BTO:0003135	zona pellucida	semapv:LexicalMatching	0.7
UBERON:0000088	trophoblast	skos:exactMatch	BTO:0001079	trophoblast	semapv:LexicalMatching	0.7
UBERON:0000115	lung epithelium	skos:exactMatch	BTO:0001653	lung epithelium	semapv:LexicalMatching	0.7
UBERON:0000118	lung bud	skos:exactMatch	BTO:0001643	lung bud	semapv:LexicalMatching	0.7
UBERON:0000121	perineurium	skos:exactMatch	BTO:0003153	perineurium	semapv:LexicalMatching	0.7
UBERON:0000124	epineurium	skos:exactMatch	BTO:0003154	epineurium	semapv:LexicalMatching	0.7
UBERON:0000156	theca externa	skos:exactMatch	BTO:0002852	theca externa	semapv:LexicalMatching	0.7
UBERON:0000157	theca interna	skos:exactMatch	BTO:0002851	theca interna	semapv:LexicalMatching	0.7
UBERON:0000159	anal canal	skos:exactMatch	BTO:0001978	anal canal	semapv:LexicalMatching	0.7
UBERON:0000160	intestine	skos:exactMatch	BTO:0000648	intestine	semapv:LexicalMatching	0.7
UBERON:0000165	mouth	skos:exactMatch	BTO:0001090	mouth	semapv:LexicalMatching	0.7
UBERON:0000173	amniotic fluid	skos:exactMatch	BTO:0000068	amniotic fluid	semapv:LexicalMatching	0.7
UBERON:0000178	blood	skos:exactMatch	BTO:0000089	blood	semapv:LexicalMatching	0.7
UBERON:0000304	tendon sheath	skos:exactMatch	BTO:0000051	tendon sheath	semapv:LexicalMatching	0.7
UBERON:0000305	amnion	skos:exactMatch	BTO:0000065	amnion	semapv:LexicalMatching	0.7
UBERON:0000307	blastula	skos:exactMatch	BTO:0000128	blastula	semapv:LexicalMatching	0.7
UBERON:0000309	body wall	skos:exactMatch	BTO:0000139	body wall	semapv:LexicalMatching	0.7
UBERON:0000310	breast	skos:exactMatch	BTO:0000149	breast	semapv:LexicalMatching	0.7
UBERON:0000314	cecum mucosa	skos:exactMatch	BTO:0000213	cecum mucosa	semapv:LexicalMatching	0.7
UBERON:0000315	subarachnoid space	skos:exactMatch	BTO:0000230	subarachnoid space	semapv:LexicalMatching	0.7
UBERON:0000316	cervical mucus	skos:exactMatch	BTO:0000242	cervical mucus	semapv:LexicalMatching	0.7
UBERON:0000317	colonic mucosa	skos:exactMatch	BTO:0000271	colonic mucosa	semapv:LexicalMatching	0.7
UBERON:0000320	duodenal mucosa	skos:exactMatch	BTO:0000367	duodenal mucosa	semapv:LexicalMatching	0.7
UBERON:0000325	gastric gland	skos:exactMatch	BTO:0000503	gastric gland	semapv:LexicalMatching	0.7
UBERON:0000326	pancreatic juice	skos:exactMatch	BTO:0000504	pancreatic juice	semapv:LexicalMatching	0.7
UBERON:0000328	gut wall	skos:exactMatch	BTO:0000547	gut wall	semapv:LexicalMatching	0.7
UBERON:0000331	ileal mucosa	skos:exactMatch	BTO:0000619	ileal mucosa	semapv:LexicalMatching	0.7
UBERON:0000332	yellow bone marrow	skos:exactMatch	BTO:0000635	yellow bone marrow	semapv:LexicalMatching	0.7
UBERON:0000333	intestinal gland	skos:exactMatch	BTO:0000640	intestinal gland	semapv:LexicalMatching	0.7
UBERON:0000344	mucosa	skos:exactMatch	BTO:0000886	mucosa	semapv:LexicalMatching	0.7
UBERON:0000348	ophthalmic nerve	skos:exactMatch	BTO:0000926	ophthalmic nerve	semapv:LexicalMatching	0.7
UBERON:0000349	limbic system	skos:exactMatch	BTO:0000928	limbic system	semapv:LexicalMatching	0.7
UBERON:0000351	nuchal ligament	skos:exactMatch	BTO:0000952	nuchal ligament	semapv:LexicalMatching	0.7
UBERON:0000353	parenchyma	skos:exactMatch	BTO:0001539	parenchyma	semapv:LexicalMatching	0.7
UBERON:0000355	pharyngeal mucosa	skos:exactMatch	BTO:0001047	pharyngeal mucosa	semapv:LexicalMatching	0.7
UBERON:0000358	blastocyst	skos:exactMatch	BTO:0001099	blastocyst	semapv:LexicalMatching	0.7
UBERON:0000361	red bone marrow	skos:exactMatch	BTO:0001160	red bone marrow	semapv:LexicalMatching	0.7
UBERON:0000362	renal medulla	skos:exactMatch	BTO:0001167	renal medulla	semapv:LexicalMatching	0.7
UBERON:0000369	corpus striatum	skos:exactMatch	BTO:0001311	corpus striatum	semapv:LexicalMatching	0.7
UBERON:0000371	syncytiotrophoblast	skos:exactMatch	BTO:0001335	syncytiotrophoblast	semapv:LexicalMatching	0.7
UBERON:0000375	mandibular nerve	skos:exactMatch	BTO:0001375	mandibular nerve	semapv:LexicalMatching	0.7
UBERON:0000377	maxillary nerve	skos:exactMatch	BTO:0001378	maxillary nerve	semapv:LexicalMatching	0.7
UBERON:0000378	tongue muscle	skos:exactMatch	BTO:0001386	tongue muscle	semapv:LexicalMatching	0.7
UBERON:0000379	tracheal mucosa	skos:exactMatch	BTO:0001390	tracheal mucosa	semapv:LexicalMatching	0.7
UBERON:0000382	apocrine sweat gland	skos:exactMatch	BTO:0001458	apocrine sweat gland	semapv:LexicalMatching	0.7
UBERON:0000389	lens cortex	skos:exactMatch	BTO:0001632	lens cortex	semapv:LexicalMatching	0.7
UBERON:0000390	lens nucleus	skos:exactMatch	BTO:0001633	lens nucleus	semapv:LexicalMatching	0.7
UBERON:0000391	leptomeninx	skos:exactMatch	BTO:0001634	leptomeninx	semapv:LexicalMatching	0.7
UBERON:0000395	cochlear ganglion	skos:exactMatch	BTO:0001688	cochlear ganglion	semapv:LexicalMatching	0.7
UBERON:0000397	colonic epithelium	skos:exactMatch	BTO:0001709	colonic epithelium	semapv:LexicalMatching	0.7
UBERON:0000399	jejunal mucosa	skos:exactMatch	BTO:0001742	jejunal mucosa	semapv:LexicalMatching	0.7
UBERON:0000400	jejunal epithelium	skos:exactMatch	BTO:0001743	jejunal epithelium	semapv:LexicalMatching	0.7
UBERON:0000402	nasal vestibule	skos:exactMatch	BTO:0001761	nasal vestibule	semapv:LexicalMatching	0.7
UBERON:0000403	scalp	skos:exactMatch	BTO:0001809	scalp	semapv:LexicalMatching	0.7
UBERON:0000409	serous gland	skos:exactMatch	BTO:0001837	serous gland	semapv:LexicalMatching	0.7
UBERON:0000410	bronchial mucosa	skos:exactMatch	BTO:0001846	bronchial mucosa	semapv:LexicalMatching	0.7
UBERON:0000412	dermal papilla	skos:exactMatch	BTO:0001858	dermal papilla	semapv:LexicalMatching	0.7
UBERON:0000414	mucous gland	skos:exactMatch	BTO:0001979	mucous gland	semapv:LexicalMatching	0.7
UBERON:0000415	artery wall	skos:exactMatch	BTO:0002009	artery wall	semapv:LexicalMatching	0.7
UBERON:0000420	myoepithelium	skos:exactMatch	BTO:0002308	myoepithelium	semapv:LexicalMatching	0.7
UBERON:0000423	eccrine sweat gland	skos:exactMatch	BTO:0002323	eccrine sweat gland	semapv:LexicalMatching	0.7
UBERON:0000424	gastric pit	skos:exactMatch	BTO:0002364	gastric pit	semapv:LexicalMatching	0.7
UBERON:0000429	enteric plexus	skos:exactMatch	BTO:0002437	enteric plexus	semapv:LexicalMatching	0.7
UBERON:0000437	arachnoid barrier layer	skos:exactMatch	BTO:0002498	arachnoid barrier layer	semapv:LexicalMatching	0.7
UBERON:0000439	arachnoid trabecula	skos:exactMatch	BTO:0002500	arachnoid trabecula	semapv:LexicalMatching	0.7
UBERON:0000440	trabecula	skos:exactMatch	BTO:0002501	trabecula	semapv:LexicalMatching	0.7
UBERON:0000442	right testicular vein	skos:exactMatch	BTO:0002679	right testicular vein	semapv:LexicalMatching	0.7
UBERON:0000443	left testicular vein	skos:exactMatch	BTO:0002680	left testicular vein	semapv:LexicalMatching	0.7
UBERON:0000444	lymphoid follicle	skos:exactMatch	BTO:0002684	lymphoid follicle	semapv:LexicalMatching	0.7
UBERON:0000454	cerebral subcortex	skos:exactMatch	BTO:0002858	cerebral subcortex	semapv:LexicalMatching	0.7
UBERON:0000457	cavernous artery	skos:exactMatch	BTO:0002996	cavernous artery	semapv:LexicalMatching	0.7
UBERON:0000458	endocervix	skos:exactMatch	BTO:0003002	endocervix	semapv:LexicalMatching	0.7
UBERON:0000459	uterine wall	skos:exactMatch	BTO:0003083	uterine wall	semapv:LexicalMatching	0.7
UBERON:0000473	testis	skos:exactMatch	BTO:0001363	testis	semapv:LexicalMatching	0.7
UBERON:0000474	female reproductive system	skos:exactMatch	BTO:0000083	female reproductive system	semapv:LexicalMatching	0.7
UBERON:0000483	epithelium	skos:exactMatch	BTO:0000416	epithelium	semapv:LexicalMatching	0.7
UBERON:0000916	abdomen	skos:exactMatch	BTO:0000020	abdomen	semapv:LexicalMatching	0.7
UBERON:0000922	embryo	skos:exactMatch	BTO:0000379	embryo	semapv:LexicalMatching	0.7
UBERON:0000923	germ layer	skos:exactMatch	BTO:0000556	germ layer	semapv:LexicalMatching	0.7
UBERON:0000924	ectoderm	skos:exactMatch	BTO:0000315	ectoderm	semapv:LexicalMatching	0.7
UBERON:0000925	endoderm	skos:exactMatch	BTO:0000800	endoderm	semapv:LexicalMatching	0.7
UBERON:0000926	mesoderm	skos:exactMatch	BTO:0000839	mesoderm	semapv:LexicalMatching	0.7
UBERON:0000930	stomodeum	skos:exactMatch	BTO:0004224	stomodeum	semapv:LexicalMatching	0.7
UBERON:0000939	imaginal disc	skos:exactMatch	BTO:0004658	imaginal disc	semapv:LexicalMatching	0.7
UBERON:0000945	stomach	skos:exactMatch	BTO:0001307	stomach	semapv:LexicalMatching	0.7
UBERON:0000947	aorta	skos:exactMatch	BTO:0000135	aorta	semapv:LexicalMatching	0.7
UBERON:0000948	heart	skos:exactMatch	BTO:0000562	heart	semapv:LexicalMatching	0.7
UBERON:0000955	brain	skos:exactMatch	BTO:0000142	brain	semapv:LexicalMatching	0.7
UBERON:0000956	cerebral cortex	skos:exactMatch	BTO:0000233	cerebral cortex	semapv:LexicalMatching	0.7
UBERON:0000961	thoracic ganglion	skos:exactMatch	BTO:0001831	thoracic ganglion	semapv:LexicalMatching	0.7
UBERON:0000964	cornea	skos:exactMatch	BTO:0000286	cornea	semapv:LexicalMatching	0.7
UBERON:0000966	retina	skos:exactMatch	BTO:0001175	retina	semapv:LexicalMatching	0.7
UBERON:0000971	ommatidium	skos:exactMatch	BTO:0001922	ommatidium	semapv:LexicalMatching	0.7
UBERON:0000974	neck	skos:exactMatch	BTO:0000420	neck	semapv:LexicalMatching	0.7
UBERON:0000975	sternum	skos:exactMatch	BTO:0001302	sternum	semapv:LexicalMatching	0.7
UBERON:0000977	pleura	skos:exactMatch	BTO:0001791	pleura	semapv:LexicalMatching	0.7
UBERON:0000979	tibia	skos:exactMatch	BTO:0001252	tibia	semapv:LexicalMatching	0.7
UBERON:0000981	femur	skos:exactMatch	BTO:0001284	femur	semapv:LexicalMatching	0.7
UBERON:0000988	pons	skos:exactMatch	BTO:0001101	pons	semapv:LexicalMatching	0.7
UBERON:0000989	penis	skos:exactMatch	BTO:0000405	penis	semapv:LexicalMatching	0.7
UBERON:0000990	reproductive system	skos:exactMatch	BTO:0000081	reproductive system	semapv:LexicalMatching	0.7
UBERON:0000991	gonad	skos:exactMatch	BTO:0000534	gonad	semapv:LexicalMatching	0.7
UBERON:0000992	ovary	skos:exactMatch	BTO:0000975	ovary	semapv:LexicalMatching	0.7
UBERON:0000993	oviduct	skos:exactMatch	BTO:0000980	oviduct	semapv:LexicalMatching	0.7
UBERON:0000995	uterus	skos:exactMatch	BTO:0001424	uterus	semapv:LexicalMatching	0.7
UBERON:0000996	vagina	skos:exactMatch	BTO:0000243	vagina	semapv:LexicalMatching	0.7
UBERON:0000998	seminal vesicle	skos:exactMatch	BTO:0001234	seminal vesicle	semapv:LexicalMatching	0.7
UBERON:0000999	ejaculatory duct	skos:exactMatch	BTO:0001580	ejaculatory duct	semapv:LexicalMatching	0.7
UBERON:0001000	vas deferens	skos:exactMatch	BTO:0001427	vas deferens	semapv:LexicalMatching	0.7
UBERON:0001004	respiratory system	skos:exactMatch	BTO:0000203	respiratory system	semapv:LexicalMatching	0.7
UBERON:0001013	adipose tissue	skos:exactMatch	BTO:0001487	adipose tissue	semapv:LexicalMatching	0.7
UBERON:0001016	nervous system	skos:exactMatch	BTO:0001484	nervous system	semapv:LexicalMatching	0.7
UBERON:0001017	central nervous system	skos:exactMatch	BTO:0000227	central nervous system	semapv:LexicalMatching	0.7
UBERON:0001021	nerve	skos:exactMatch	BTO:0000925	nerve	semapv:LexicalMatching	0.7
UBERON:0001040	yolk sac	skos:exactMatch	BTO:0001471	yolk sac	semapv:LexicalMatching	0.7
UBERON:0001041	foregut	skos:exactMatch	BTO:0000507	foregut	semapv:LexicalMatching	0.7
UBERON:0001043	esophagus	skos:exactMatch	BTO:0000959	esophagus	semapv:LexicalMatching	0.7
UBERON:0001045	midgut	skos:exactMatch	BTO:0000863	midgut	semapv:LexicalMatching	0.7
UBERON:0001046	hindgut	skos:exactMatch	BTO:0000510	hindgut	semapv:LexicalMatching	0.7
UBERON:0001048	primordium	skos:exactMatch	BTO:0001886	primordium	semapv:LexicalMatching	0.7
UBERON:0001049	neural tube	skos:exactMatch	BTO:0001057	neural tube	semapv:LexicalMatching	0.7
UBERON:0001052	rectum	skos:exactMatch	BTO:0001158	rectum	semapv:LexicalMatching	0.7
UBERON:0001054	Malpighian tubule	skos:exactMatch	BTO:0000810	malpighian tubule	semapv:LexicalMatching	0.7
UBERON:0001058	mushroom body	skos:exactMatch	BTO:0002675	mushroom body	semapv:LexicalMatching	0.7
UBERON:0001070	external carotid artery	skos:exactMatch	BTO:0004696	external carotid artery	semapv:LexicalMatching	0.7
UBERON:0001072	inferior vena cava	skos:exactMatch	BTO:0002682	inferior vena cava	semapv:LexicalMatching	0.7
UBERON:0001087	pleural fluid	skos:exactMatch	BTO:0003080	pleural fluid	semapv:LexicalMatching	0.7
UBERON:0001088	urine	skos:exactMatch	BTO:0001419	urine	semapv:LexicalMatching	0.7
UBERON:0001089	sweat	skos:exactMatch	BTO:0001254	sweat	semapv:LexicalMatching	0.7
UBERON:0001103	diaphragm	skos:exactMatch	BTO:0000341	diaphragm	semapv:LexicalMatching	0.7
UBERON:0001132	parathyroid gland	skos:exactMatch	BTO:0000997	parathyroid gland	semapv:LexicalMatching	0.7
UBERON:0001136	mesothelium	skos:exactMatch	BTO:0002422	mesothelium	semapv:LexicalMatching	0.7
UBERON:0001137	dorsum	skos:exactMatch	BTO:0001713	dorsum	semapv:LexicalMatching	0.7
UBERON:0001138	superior mesenteric vein	skos:exactMatch	BTO:0002783	superior mesenteric vein	semapv:LexicalMatching	0.7
UBERON:0001140	renal vein	skos:exactMatch	BTO:0002681	renal vein	semapv:LexicalMatching	0.7
UBERON:0001144	testicular vein	skos:exactMatch	BTO:0002678	testicular vein	semapv:LexicalMatching	0.7
UBERON:0001154	vermiform appendix	skos:exactMatch	BTO:0000084	vermiform appendix	semapv:LexicalMatching	0.7
UBERON:0001155	colon	skos:exactMatch	BTO:0000269	colon	semapv:LexicalMatching	0.7
UBERON:0001179	peritoneal cavity	skos:exactMatch	BTO:0001782	peritoneal cavity	semapv:LexicalMatching	0.7
UBERON:0001182	superior mesenteric artery	skos:exactMatch	BTO:0002303	superior mesenteric artery	semapv:LexicalMatching	0.7
UBERON:0001183	inferior mesenteric artery	skos:exactMatch	BTO:0002302	inferior mesenteric artery	semapv:LexicalMatching	0.7
UBERON:0001184	renal artery	skos:exactMatch	BTO:0001165	renal artery	semapv:LexicalMatching	0.7
UBERON:0001193	hepatic artery	skos:exactMatch	BTO:0004307	hepatic artery	semapv:LexicalMatching	0.7
UBERON:0001212	duodenal gland	skos:exactMatch	BTO:0002376	duodenal gland	semapv:LexicalMatching	0.7
UBERON:0001215	inferior mesenteric vein	skos:exactMatch	BTO:0002782	inferior mesenteric vein	semapv:LexicalMatching	0.7
UBERON:0001228	renal papilla	skos:exactMatch	BTO:0003925	renal papilla	semapv:LexicalMatching	0.7
UBERON:0001229	renal corpuscle	skos:exactMatch	BTO:0000333	renal corpuscle	semapv:LexicalMatching	0.7
UBERON:0001235	adrenal cortex	skos:exactMatch	BTO:0000045	adrenal cortex	semapv:LexicalMatching	0.7
UBERON:0001236	adrenal medulla	skos:exactMatch	BTO:0000049	adrenal medulla	semapv:LexicalMatching	0.7
UBERON:0001242	intestinal mucosa	skos:exactMatch	BTO:0000642	intestinal mucosa	semapv:LexicalMatching	0.7
UBERON:0001245	anus	skos:exactMatch	BTO:0001680	anus	semapv:LexicalMatching	0.7
UBERON:0001255	urinary bladder	skos:exactMatch	BTO:0001418	urinary bladder	semapv:LexicalMatching	0.7
UBERON:0001264	pancreas	skos:exactMatch	BTO:0000988	pancreas	semapv:LexicalMatching	0.7
UBERON:0001268	peritoneal fluid	skos:exactMatch	BTO:0001031	peritoneal fluid	semapv:LexicalMatching	0.7
UBERON:0001277	intestinal epithelium	skos:exactMatch	BTO:0000781	intestinal epithelium	semapv:LexicalMatching	0.7
UBERON:0001285	nephron	skos:exactMatch	BTO:0000924	nephron	semapv:LexicalMatching	0.7
UBERON:0001296	myometrium	skos:exactMatch	BTO:0000907	myometrium	semapv:LexicalMatching	0.7
UBERON:0001299	glans penis	skos:exactMatch	BTO:0003118	glans penis	semapv:LexicalMatching	0.7
UBERON:0001301	epididymis	skos:exactMatch	BTO:0000408	epididymis	semapv:LexicalMatching	0.7
UBERON:0001305	ovarian follicle	skos:exactMatch	BTO:0000475	ovarian follicle	semapv:LexicalMatching	0.7
UBERON:0001308	external iliac artery	skos:exactMatch	BTO:0004666	external iliac artery	semapv:LexicalMatching	0.7
UBERON:0001309	internal iliac artery	skos:exactMatch	BTO:0004667	internal iliac artery	semapv:LexicalMatching	0.7
UBERON:0001310	umbilical artery	skos:exactMatch	BTO:0000841	umbilical artery	semapv:LexicalMatching	0.7
UBERON:0001322	sciatic nerve	skos:exactMatch	BTO:0001221	sciatic nerve	semapv:LexicalMatching	0.7
UBERON:0001333	male urethra	skos:exactMatch	BTO:0004089	male urethra	semapv:LexicalMatching	0.7
UBERON:0001334	female urethra	skos:exactMatch	BTO:0004088	female urethra	semapv:LexicalMatching	0.7
UBERON:0001335	prostatic urethra	skos:exactMatch	BTO:0003163	prostatic urethra	semapv:LexicalMatching	0.7
UBERON:0001347	white adipose tissue	skos:exactMatch	BTO:0001456	white adipose tissue	semapv:LexicalMatching	0.7
UBERON:0001348	brown adipose tissue	skos:exactMatch	BTO:0000156	brown adipose tissue	semapv:LexicalMatching	0.7
UBERON:0001359	cerebrospinal fluid	skos:exactMatch	BTO:0000237	cerebrospinal fluid	semapv:LexicalMatching	0.7
UBERON:0001363	great saphenous vein	skos:exactMatch	BTO:0003271	great saphenous vein	semapv:LexicalMatching	0.7
UBERON:0001434	skeletal system	skos:exactMatch	BTO:0001486	skeletal system	semapv:LexicalMatching	0.7
UBERON:0001437	epiphysis	skos:exactMatch	BTO:0000413	epiphysis	semapv:LexicalMatching	0.7
UBERON:0001446	fibula	skos:exactMatch	BTO:0002346	fibula	semapv:LexicalMatching	0.7
UBERON:0001447	tarsal bone	skos:exactMatch	BTO:0002343	tarsal bone	semapv:LexicalMatching	0.7
UBERON:0001448	metatarsal bone	skos:exactMatch	BTO:0002347	metatarsal bone	semapv:LexicalMatching	0.7
UBERON:0001456	face	skos:exactMatch	BTO:0003369	face	semapv:LexicalMatching	0.7
UBERON:0001465	knee	skos:exactMatch	BTO:0003595	knee	semapv:LexicalMatching	0.7
UBERON:0001488	ankle joint	skos:exactMatch	BTO:0004706	ankle joint	semapv:LexicalMatching	0.7
UBERON:0001495	pectoral muscle	skos:exactMatch	BTO:0000023	pectoral muscle	semapv:LexicalMatching	0.7
UBERON:0001516	abdominal aorta	skos:exactMatch	BTO:0002976	abdominal aorta	semapv:LexicalMatching	0.7
UBERON:0001532	internal carotid artery	skos:exactMatch	BTO:0004697	internal carotid artery	semapv:LexicalMatching	0.7
UBERON:0001547	small saphenous vein	skos:exactMatch	BTO:0003272	small saphenous vein	semapv:LexicalMatching	0.7
UBERON:0001567	cheek	skos:exactMatch	BTO:0001754	cheek	semapv:LexicalMatching	0.7
UBERON:0001579	olfactory nerve	skos:exactMatch	BTO:0003648	olfactory nerve	semapv:LexicalMatching	0.7
UBERON:0001605	ciliary muscle	skos:exactMatch	BTO:0000654	ciliary muscle	semapv:LexicalMatching	0.7
UBERON:0001621	coronary artery	skos:exactMatch	BTO:0000290	coronary artery	semapv:LexicalMatching	0.7
UBERON:0001637	artery	skos:exactMatch	BTO:0000573	artery	semapv:LexicalMatching	0.7
UBERON:0001638	vein	skos:exactMatch	BTO:0000234	vein	semapv:LexicalMatching	0.7
UBERON:0001645	trigeminal nerve	skos:exactMatch	BTO:0001072	trigeminal nerve	semapv:LexicalMatching	0.7
UBERON:0001648	vestibulocochlear nerve	skos:exactMatch	BTO:0003490	vestibulocochlear nerve	semapv:LexicalMatching	0.7
UBERON:0001649	glossopharyngeal nerve	skos:exactMatch	BTO:0004979	glossopharyngeal nerve	semapv:LexicalMatching	0.7
UBERON:0001650	hypoglossal nerve	skos:exactMatch	BTO:0003386	hypoglossal nerve	semapv:LexicalMatching	0.7
UBERON:0001675	trigeminal ganglion	skos:exactMatch	BTO:0001231	trigeminal ganglion	semapv:LexicalMatching	0.7
UBERON:0001679	ethmoid bone	skos:exactMatch	BTO:0004140	ethmoid bone	semapv:LexicalMatching	0.7
UBERON:0001707	nasal cavity	skos:exactMatch	BTO:0002096	nasal cavity	semapv:LexicalMatching	0.7
UBERON:0001711	eyelid	skos:exactMatch	BTO:0002241	eyelid	semapv:LexicalMatching	0.7
UBERON:0001714	cranial ganglion	skos:exactMatch	BTO:0000106	cranial ganglion	semapv:LexicalMatching	0.7
UBERON:0001723	tongue	skos:exactMatch	BTO:0001385	tongue	semapv:LexicalMatching	0.7
UBERON:0001727	taste bud	skos:exactMatch	BTO:0000989	taste bud	semapv:LexicalMatching	0.7
UBERON:0001728	nasopharynx	skos:exactMatch	BTO:0000662	nasopharynx	semapv:LexicalMatching	0.7
UBERON:0001736	submandibular gland	skos:exactMatch	BTO:0001316	submandibular gland	semapv:LexicalMatching	0.7
UBERON:0001737	larynx	skos:exactMatch	BTO:0001208	larynx	semapv:LexicalMatching	0.7
UBERON:0001739	laryngeal cartilage	skos:exactMatch	BTO:0003660	laryngeal cartilage	semapv:LexicalMatching	0.7
UBERON:0001744	lymphoid tissue	skos:exactMatch	BTO:0000753	lymphoid tissue	semapv:LexicalMatching	0.7
UBERON:0001747	parenchyma of thyroid gland	skos:exactMatch	BTO:0004579	parenchyma of thyroid gland	semapv:LexicalMatching	0.7
UBERON:0001753	cementum	skos:exactMatch	BTO:0002525	cementum	semapv:LexicalMatching	0.7
UBERON:0001754	dental pulp	skos:exactMatch	BTO:0000339	dental pulp	semapv:LexicalMatching	0.7
UBERON:0001756	middle ear	skos:exactMatch	BTO:0002099	middle ear	semapv:LexicalMatching	0.7
UBERON:0001758	periodontium	skos:exactMatch	BTO:0001021	periodontium	semapv:LexicalMatching	0.7
UBERON:0001759	vagus nerve	skos:exactMatch	BTO:0003472	vagus nerve	semapv:LexicalMatching	0.7
UBERON:0001765	mammary duct	skos:exactMatch	BTO:0002845	mammary duct	semapv:LexicalMatching	0.7
UBERON:0001769	iris	skos:exactMatch	BTO:0000653	iris	semapv:LexicalMatching	0.7
UBERON:0001772	corneal epithelium	skos:exactMatch	BTO:0000287	corneal epithelium	semapv:LexicalMatching	0.7
UBERON:0001773	sclera	skos:exactMatch	BTO:0001606	sclera	semapv:LexicalMatching	0.7
UBERON:0001775	ciliary body	skos:exactMatch	BTO:0000260	ciliary body	semapv:LexicalMatching	0.7
UBERON:0001778	ciliary epithelium	skos:exactMatch	BTO:0001770	ciliary epithelium	semapv:LexicalMatching	0.7
UBERON:0001780	spinal nerve	skos:exactMatch	BTO:0000870	spinal nerve	semapv:LexicalMatching	0.7
UBERON:0001785	cranial nerve	skos:exactMatch	BTO:0001104	cranial nerve	semapv:LexicalMatching	0.7
UBERON:0001797	vitreous humor	skos:exactMatch	BTO:0001451	vitreous humor	semapv:LexicalMatching	0.7
UBERON:0001800	sensory ganglion	skos:exactMatch	BTO:0005630	sensory ganglion	semapv:LexicalMatching	0.7
UBERON:0001806	sympathetic ganglion	skos:exactMatch	BTO:0001333	sympathetic ganglion	semapv:LexicalMatching	0.7
UBERON:0001808	parasympathetic ganglion	skos:exactMatch	BTO:0001256	parasympathetic ganglion	semapv:LexicalMatching	0.7
UBERON:0001810	nerve plexus	skos:exactMatch	BTO:0000205	nerve plexus	semapv:LexicalMatching	0.7
UBERON:0001811	conjunctiva	skos:exactMatch	BTO:0003415	conjunctiva	semapv:LexicalMatching	0.7
UBERON:0001820	sweat gland	skos:exactMatch	BTO:0001331	sweat gland	semapv:LexicalMatching	0.7
UBERON:0001821	sebaceous gland	skos:exactMatch	BTO:0001980	sebaceous gland	semapv:LexicalMatching	0.7
UBERON:0001828	gingiva	skos:exactMatch	BTO:0000519	gingiva	semapv:LexicalMatching	0.7
UBERON:0001831	parotid gland	skos:exactMatch	BTO:0001004	parotid gland	semapv:LexicalMatching	0.7
UBERON:0001832	sublingual gland	skos:exactMatch	BTO:0001315	sublingual gland	semapv:LexicalMatching	0.7
UBERON:0001836	saliva	skos:exactMatch	BTO:0001202	saliva	semapv:LexicalMatching	0.7
UBERON:0001839	bony labyrinth	skos:exactMatch	BTO:0004685	bony labyrinth	semapv:LexicalMatching	0.7
UBERON:0001840	semicircular canal	skos:exactMatch	BTO:0003383	semicircular canal	semapv:LexicalMatching	0.7
UBERON:0001844	cochlea	skos:exactMatch	BTO:0000267	cochlea	semapv:LexicalMatching	0.7
UBERON:0001862	vestibular labyrinth	skos:exactMatch	BTO:0001856	vestibular labyrinth	semapv:LexicalMatching	0.7
UBERON:0001866	sebum	skos:exactMatch	BTO:0001981	sebum	semapv:LexicalMatching	0.7
UBERON:0001869	cerebral hemisphere	skos:exactMatch	BTO:0000231	cerebral hemisphere	semapv:LexicalMatching	0.7
UBERON:0001871	temporal lobe	skos:exactMatch	BTO:0001355	temporal lobe	semapv:LexicalMatching	0.7
UBERON:0001872	parietal lobe	skos:exactMatch	BTO:0001001	parietal lobe	semapv:LexicalMatching	0.7
UBERON:0001873	caudate nucleus	skos:exactMatch	BTO:0000211	caudate nucleus	semapv:LexicalMatching	0.7
UBERON:0001875	globus pallidus	skos:exactMatch	BTO:0002246	globus pallidus	semapv:LexicalMatching	0.7
UBERON:0001876	amygdala	skos:exactMatch	BTO:0001042	amygdala	semapv:LexicalMatching	0.7
UBERON:0001882	nucleus accumbens	skos:exactMatch	BTO:0001862	nucleus accumbens	semapv:LexicalMatching	0.7
UBERON:0001883	olfactory tubercle	skos:exactMatch	BTO:0001869	olfactory tubercle	semapv:LexicalMatching	0.7
UBERON:0001884	phrenic nerve	skos:exactMatch	BTO:0001063	phrenic nerve	semapv:LexicalMatching	0.7
UBERON:0001886	choroid plexus	skos:exactMatch	BTO:0000258	choroid plexus	semapv:LexicalMatching	0.7
UBERON:0001890	forebrain	skos:exactMatch	BTO:0000478	forebrain	semapv:LexicalMatching	0.7
UBERON:0001891	midbrain	skos:exactMatch	BTO:0000138	midbrain	semapv:LexicalMatching	0.7
UBERON:0001893	telencephalon	skos:exactMatch	BTO:0000239	telencephalon	semapv:LexicalMatching	0.7
UBERON:0001894	diencephalon	skos:exactMatch	BTO:0000342	diencephalon	semapv:LexicalMatching	0.7
UBERON:0001895	metencephalon	skos:exactMatch	BTO:0000673	metencephalon	semapv:LexicalMatching	0.7
UBERON:0001896	medulla oblongata	skos:exactMatch	BTO:0000041	medulla oblongata	semapv:LexicalMatching	0.7
UBERON:0001898	hypothalamus	skos:exactMatch	BTO:0000614	hypothalamus	semapv:LexicalMatching	0.7
UBERON:0001899	epithalamus	skos:exactMatch	BTO:0000175	epithalamus	semapv:LexicalMatching	0.7
UBERON:0001906	subthalamic nucleus	skos:exactMatch	BTO:0002252	subthalamic nucleus	semapv:LexicalMatching	0.7
UBERON:0001907	zona incerta	skos:exactMatch	BTO:0003146	zona incerta	semapv:LexicalMatching	0.7
UBERON:0001911	mammary gland	skos:exactMatch	BTO:0000817	mammary gland	semapv:LexicalMatching	0.7
UBERON:0001913	milk	skos:exactMatch	BTO:0000868	milk	semapv:LexicalMatching	0.7
UBERON:0001926	lateral geniculate body	skos:exactMatch	BTO:0004367	lateral geniculate body	semapv:LexicalMatching	0.7
UBERON:0001928	preoptic area	skos:exactMatch	BTO:0001796	preoptic area	semapv:LexicalMatching	0.7
UBERON:0001930	paraventricular nucleus of hypothalamus	skos:exactMatch	BTO:0002476	paraventricular nucleus of hypothalamus	semapv:LexicalMatching	0.7
UBERON:0001949	gingival epithelium	skos:exactMatch	BTO:0004998	gingival epithelium	semapv:LexicalMatching	0.7
UBERON:0001950	neocortex	skos:exactMatch	BTO:0000920	neocortex	semapv:LexicalMatching	0.7
UBERON:0001968	semen	skos:exactMatch	BTO:0001230	semen	semapv:LexicalMatching	0.7
UBERON:0001969	blood plasma	skos:exactMatch	BTO:0000131	blood plasma	semapv:LexicalMatching	0.7
UBERON:0001970	bile	skos:exactMatch	BTO:0000121	bile	semapv:LexicalMatching	0.7
UBERON:0001971	gastric juice	skos:exactMatch	BTO:0000501	gastric juice	semapv:LexicalMatching	0.7
UBERON:0001979	venule	skos:exactMatch	BTO:0002626	venule	semapv:LexicalMatching	0.7
UBERON:0001980	arteriole	skos:exactMatch	BTO:0001997	arteriole	semapv:LexicalMatching	0.7
UBERON:0001981	blood vessel	skos:exactMatch	BTO:0001102	blood vessel	semapv:LexicalMatching	0.7
UBERON:0001982	capillary	skos:exactMatch	BTO:0002045	capillary	semapv:LexicalMatching	0.7
UBERON:0001986	endothelium	skos:exactMatch	BTO:0000393	endothelium	semapv:LexicalMatching	0.7
UBERON:0001987	placenta	skos:exactMatch	BTO:0001078	placenta	semapv:LexicalMatching	0.7
UBERON:0001988	feces	skos:exactMatch	BTO:0000440	feces	semapv:LexicalMatching	0.7
UBERON:0001989	superior cervical ganglion	skos:exactMatch	BTO:0001325	superior cervical ganglion	semapv:LexicalMatching	0.7
UBERON:0001990	middle cervical ganglion	skos:exactMatch	BTO:0000359	middle cervical ganglion	semapv:LexicalMatching	0.7
UBERON:0001991	cervical ganglion	skos:exactMatch	BTO:0000113	cervical ganglion	semapv:LexicalMatching	0.7
UBERON:0001997	olfactory epithelium	skos:exactMatch	BTO:0000108	olfactory epithelium	semapv:LexicalMatching	0.7
UBERON:0002005	enteric nervous system	skos:exactMatch	BTO:0002506	enteric nervous system	semapv:LexicalMatching	0.7
UBERON:0002012	pulmonary artery	skos:exactMatch	BTO:0000778	pulmonary artery	semapv:LexicalMatching	0.7
UBERON:0002016	pulmonary vein	skos:exactMatch	BTO:0001799	pulmonary vein	semapv:LexicalMatching	0.7
UBERON:0002017	portal vein	skos:exactMatch	BTO:0001792	portal vein	semapv:LexicalMatching	0.7
UBERON:0002028	hindbrain	skos:exactMatch	BTO:0000672	hindbrain	semapv:LexicalMatching	0.7
UBERON:0002030	nipple	skos:exactMatch	BTO:0000821	nipple	semapv:LexicalMatching	0.7
UBERON:0002034	suprachiasmatic nucleus	skos:exactMatch	BTO:0001822	suprachiasmatic nucleus	semapv:LexicalMatching	0.7
UBERON:0002037	cerebellum	skos:exactMatch	BTO:0000232	cerebellum	semapv:LexicalMatching	0.7
UBERON:0002038	substantia nigra	skos:exactMatch	BTO:0000143	substantia nigra	semapv:LexicalMatching	0.7
UBERON:0002046	thyroid gland	skos:exactMatch	BTO:0001379	thyroid gland	semapv:LexicalMatching	0.7
UBERON:0002048	lung	skos:exactMatch	BTO:0000763	lung	semapv:LexicalMatching	0.7
UBERON:0002049	vasculature	skos:exactMatch	BTO:0003718	vasculature	semapv:LexicalMatching	0.7
UBERON:0002050	embryonic structure	skos:exactMatch	BTO:0000174	embryonic structure	semapv:LexicalMatching	0.7
UBERON:0002059	submandibular ganglion	skos:exactMatch	BTO:0001318	submandibular ganglion	semapv:LexicalMatching	0.7
UBERON:0002060	femoral artery	skos:exactMatch	BTO:0001624	femoral artery	semapv:LexicalMatching	0.7
UBERON:0002066	umbilical vein	skos:exactMatch	BTO:0001509	umbilical vein	semapv:LexicalMatching	0.7
UBERON:0002067	dermis	skos:exactMatch	BTO:0000294	dermis	semapv:LexicalMatching	0.7
UBERON:0002072	hypodermis	skos:exactMatch	BTO:0000313	hypodermis	semapv:LexicalMatching	0.7
UBERON:0002073	hair follicle	skos:exactMatch	BTO:0000554	hair follicle	semapv:LexicalMatching	0.7
UBERON:0002074	hair shaft	skos:exactMatch	BTO:0004672	hair shaft	semapv:LexicalMatching	0.7
UBERON:0002075	viscus	skos:exactMatch	BTO:0001491	viscus	semapv:LexicalMatching	0.7
UBERON:0002094	interventricular septum	skos:exactMatch	BTO:0002483	interventricular septum	semapv:LexicalMatching	0.7
UBERON:0002095	mesentery	skos:exactMatch	BTO:0001380	mesentery	semapv:LexicalMatching	0.7
UBERON:0002100	trunk	skos:exactMatch	BTO:0001493	trunk	semapv:LexicalMatching	0.7
UBERON:0002101	limb	skos:exactMatch	BTO:0001492	limb	semapv:LexicalMatching	0.7
UBERON:0002102	forelimb	skos:exactMatch	BTO:0001729	forelimb	semapv:LexicalMatching	0.7
UBERON:0002103	hindlimb	skos:exactMatch	BTO:0002345	hindlimb	semapv:LexicalMatching	0.7
UBERON:0002106	spleen	skos:exactMatch	BTO:0001281	spleen	semapv:LexicalMatching	0.7
UBERON:0002107	liver	skos:exactMatch	BTO:0000759	liver	semapv:LexicalMatching	0.7
UBERON:0002108	small intestine	skos:exactMatch	BTO:0000651	small intestine	semapv:LexicalMatching	0.7
UBERON:0002113	kidney	skos:exactMatch	BTO:0000671	kidney	semapv:LexicalMatching	0.7
UBERON:0002114	duodenum	skos:exactMatch	BTO:0000365	duodenum	semapv:LexicalMatching	0.7
UBERON:0002115	jejunum	skos:exactMatch	BTO:0000657	jejunum	semapv:LexicalMatching	0.7
UBERON:0002116	ileum	skos:exactMatch	BTO:0000620	ileum	semapv:LexicalMatching	0.7
UBERON:0002120	pronephros	skos:exactMatch	BTO:0001541	pronephros	semapv:LexicalMatching	0.7
UBERON:0002129	cerebellar cortex	skos:exactMatch	BTO:0000043	cerebellar cortex	semapv:LexicalMatching	0.7
UBERON:0002139	subcommissural organ	skos:exactMatch	BTO:0001820	subcommissural organ	semapv:LexicalMatching	0.7
UBERON:0002148	locus ceruleus	skos:exactMatch	BTO:0001408	locus ceruleus	semapv:LexicalMatching	0.7
UBERON:0002162	area postrema	skos:exactMatch	BTO:0003425	area postrema	semapv:LexicalMatching	0.7
UBERON:0002165	endocardium	skos:exactMatch	BTO:0000387	endocardium	semapv:LexicalMatching	0.7
UBERON:0002169	alveolar sac	skos:exactMatch	BTO:0000061	alveolar sac	semapv:LexicalMatching	0.7
UBERON:0002185	bronchus	skos:exactMatch	BTO:0001340	bronchus	semapv:LexicalMatching	0.7
UBERON:0002186	bronchiole	skos:exactMatch	BTO:0002375	bronchiole	semapv:LexicalMatching	0.7
UBERON:0002187	terminal bronchiole	skos:exactMatch	BTO:0003223	terminal bronchiole	semapv:LexicalMatching	0.7
UBERON:0002188	respiratory bronchiole	skos:exactMatch	BTO:0003222	respiratory bronchiole	semapv:LexicalMatching	0.7
UBERON:0002190	subcutaneous adipose tissue	skos:exactMatch	BTO:0004042	subcutaneous adipose tissue	semapv:LexicalMatching	0.7
UBERON:0002191	subiculum	skos:exactMatch	BTO:0003087	subiculum	semapv:LexicalMatching	0.7
UBERON:0002196	adenohypophysis	skos:exactMatch	BTO:0000040	adenohypophysis	semapv:LexicalMatching	0.7
UBERON:0002198	neurohypophysis	skos:exactMatch	BTO:0000937	neurohypophysis	semapv:LexicalMatching	0.7
UBERON:0002199	integument	skos:exactMatch	BTO:0000634	integument	semapv:LexicalMatching	0.7
UBERON:0002219	subfornical organ	skos:exactMatch	BTO:0006266	subfornical organ	semapv:LexicalMatching	0.7
UBERON:0002222	perichondrium	skos:exactMatch	BTO:0005089	perichondrium	semapv:LexicalMatching	0.7
UBERON:0002240	spinal cord	skos:exactMatch	BTO:0001279	spinal cord	semapv:LexicalMatching	0.7
UBERON:0002242	nucleus pulposus	skos:exactMatch	BTO:0003626	nucleus pulposus	semapv:LexicalMatching	0.7
UBERON:0002255	vomeronasal organ	skos:exactMatch	BTO:0002608	vomeronasal organ	semapv:LexicalMatching	0.7
UBERON:0002262	celiac ganglion	skos:exactMatch	BTO:0006296	celiac ganglion	semapv:LexicalMatching	0.7
UBERON:0002264	olfactory bulb	skos:exactMatch	BTO:0000961	olfactory bulb	semapv:LexicalMatching	0.7
UBERON:0002265	olfactory tract	skos:exactMatch	BTO:0003647	olfactory tract	semapv:LexicalMatching	0.7
UBERON:0002266	anterior olfactory nucleus	skos:exactMatch	BTO:0003649	anterior olfactory nucleus	semapv:LexicalMatching	0.7
UBERON:0002268	olfactory organ	skos:exactMatch	BTO:0001772	olfactory organ	semapv:LexicalMatching	0.7
UBERON:0002278	perilymphatic space	skos:exactMatch	BTO:0002083	perilymphatic space	semapv:LexicalMatching	0.7
UBERON:0002303	juxtaglomerular apparatus	skos:exactMatch	BTO:0005157	juxtaglomerular apparatus	semapv:LexicalMatching	0.7
UBERON:0002319	mesangium	skos:exactMatch	BTO:0002494	mesangium	semapv:LexicalMatching	0.7
UBERON:0002328	notochord	skos:exactMatch	BTO:0001768	notochord	semapv:LexicalMatching	0.7
UBERON:0002329	somite	skos:exactMatch	BTO:0001558	somite	semapv:LexicalMatching	0.7
UBERON:0002331	umbilical cord	skos:exactMatch	BTO:0001415	umbilical cord	semapv:LexicalMatching	0.7
UBERON:0002335	macula densa	skos:exactMatch	BTO:0003940	macula densa	semapv:LexicalMatching	0.7
UBERON:0002336	corpus callosum	skos:exactMatch	BTO:0000615	corpus callosum	semapv:LexicalMatching	0.7
UBERON:0002342	neural crest	skos:exactMatch	BTO:0001764	neural crest	semapv:LexicalMatching	0.7
UBERON:0002349	myocardium	skos:exactMatch	BTO:0000901	myocardium	semapv:LexicalMatching	0.7
UBERON:0002352	atrioventricular node	skos:exactMatch	BTO:0005689	atrioventricular node	semapv:LexicalMatching	0.7
UBERON:0002354	cardiac Purkinje fiber	skos:exactMatch	BTO:0001735	cardiac Purkinje fiber	semapv:LexicalMatching	0.7
UBERON:0002358	peritoneum	skos:exactMatch	BTO:0001472	peritoneum	semapv:LexicalMatching	0.7
UBERON:0002360	meninx	skos:exactMatch	BTO:0000144	meninx	semapv:LexicalMatching	0.7
UBERON:0002361	pia mater	skos:exactMatch	BTO:0001635	pia mater	semapv:LexicalMatching	0.7
UBERON:0002362	arachnoid mater	skos:exactMatch	BTO:0001636	arachnoid mater	semapv:LexicalMatching	0.7
UBERON:0002363	dura mater	skos:exactMatch	BTO:0001637	dura mater	semapv:LexicalMatching	0.7
UBERON:0002365	exocrine gland	skos:exactMatch	BTO:0000765	exocrine gland	semapv:LexicalMatching	0.7
UBERON:0002367	prostate gland	skos:exactMatch	BTO:0001129	prostate gland	semapv:LexicalMatching	0.7
UBERON:0002368	endocrine gland	skos:exactMatch	BTO:0001488	endocrine gland	semapv:LexicalMatching	0.7
UBERON:0002369	adrenal gland	skos:exactMatch	BTO:0000047	adrenal gland	semapv:LexicalMatching	0.7
UBERON:0002370	thymus	skos:exactMatch	BTO:0001374	thymus	semapv:LexicalMatching	0.7
UBERON:0002371	bone marrow	skos:exactMatch	BTO:0000141	bone marrow	semapv:LexicalMatching	0.7
UBERON:0002372	tonsil	skos:exactMatch	BTO:0001387	tonsil	semapv:LexicalMatching	0.7
UBERON:0002384	connective tissue	skos:exactMatch	BTO:0000421	connective tissue	semapv:LexicalMatching	0.7
UBERON:0002390	hematopoietic system	skos:exactMatch	BTO:0000570	hematopoietic system	semapv:LexicalMatching	0.7
UBERON:0002391	lymph	skos:exactMatch	BTO:0000855	lymph	semapv:LexicalMatching	0.7
UBERON:0002394	bile duct	skos:exactMatch	BTO:0000122	bile duct	semapv:LexicalMatching	0.7
UBERON:0002402	pleural cavity	skos:exactMatch	BTO:0004422	pleural cavity	semapv:LexicalMatching	0.7
UBERON:0002405	immune system	skos:exactMatch	BTO:0005810	immune system	semapv:LexicalMatching	0.7
UBERON:0002407	pericardium	skos:exactMatch	BTO:0000717	pericardium	semapv:LexicalMatching	0.7
UBERON:0002409	pericardial fluid	skos:exactMatch	BTO:0001016	pericardial fluid	semapv:LexicalMatching	0.7
UBERON:0002410	autonomic nervous system	skos:exactMatch	BTO:0002507	autonomic nervous system	semapv:LexicalMatching	0.7
UBERON:0002411	clitoris	skos:exactMatch	BTO:0002020	clitoris	semapv:LexicalMatching	0.7
UBERON:0002412	vertebra	skos:exactMatch	BTO:0006196	vertebra	semapv:LexicalMatching	0.7
UBERON:0002415	tail	skos:exactMatch	BTO:0001348	tail	semapv:LexicalMatching	0.7
UBERON:0002420	basal ganglion	skos:exactMatch	BTO:0000235	basal ganglion	semapv:LexicalMatching	0.7
UBERON:0002422	fourth ventricle	skos:exactMatch	BTO:0003426	fourth ventricle	semapv:LexicalMatching	0.7
UBERON:0002424	oral epithelium	skos:exactMatch	BTO:0001775	oral epithelium	semapv:LexicalMatching	0.7
UBERON:0002444	lens fiber	skos:exactMatch	BTO:0000724	lens fiber	semapv:LexicalMatching	0.7
UBERON:0002450	decidua	skos:exactMatch	BTO:0001360	decidua	semapv:LexicalMatching	0.7
UBERON:0002451	endometrial gland	skos:exactMatch	BTO:0003433	endometrial gland	semapv:LexicalMatching	0.7
UBERON:0002464	nerve trunk	skos:exactMatch	BTO:0000927	nerve trunk	semapv:LexicalMatching	0.7
UBERON:0002495	long bone	skos:exactMatch	BTO:0004256	long bone	semapv:LexicalMatching	0.7
UBERON:0002499	cochlear labyrinth	skos:exactMatch	BTO:0004686	cochlear labyrinth	semapv:LexicalMatching	0.7
UBERON:0002509	mesenteric lymph node	skos:exactMatch	BTO:0000767	mesenteric lymph node	semapv:LexicalMatching	0.7
UBERON:0002512	corpus luteum	skos:exactMatch	BTO:0000292	corpus luteum	semapv:LexicalMatching	0.7
UBERON:0002513	endochondral bone	skos:exactMatch	BTO:0002157	endochondral bone	semapv:LexicalMatching	0.7
UBERON:0002515	periosteum	skos:exactMatch	BTO:0001022	periosteum	semapv:LexicalMatching	0.7
UBERON:0002530	gland	skos:exactMatch	BTO:0000522	gland	semapv:LexicalMatching	0.7
UBERON:0002548	larva	skos:exactMatch	BTO:0000707	larva	semapv:LexicalMatching	0.7
UBERON:0002707	corticospinal tract	skos:exactMatch	BTO:0006132	corticospinal tract	semapv:LexicalMatching	0.7
UBERON:0002708	posterior periventricular nucleus	skos:exactMatch	BTO:0002479	posterior periventricular nucleus	semapv:LexicalMatching	0.7
UBERON:0002709	posterior nuclear complex of thalamus	skos:exactMatch	BTO:0002462	posterior nuclear complex of thalamus	semapv:LexicalMatching	0.7
UBERON:0002714	rubrospinal tract	skos:exactMatch	BTO:0006159	rubrospinal tract	semapv:LexicalMatching	0.7
UBERON:0002743	basal forebrain	skos:exactMatch	BTO:0002444	basal forebrain	semapv:LexicalMatching	0.7
UBERON:0002886	lateral amygdaloid nucleus	skos:exactMatch	BTO:0002704	lateral amygdaloid nucleus	semapv:LexicalMatching	0.7
UBERON:0002892	medial amygdaloid nucleus	skos:exactMatch	BTO:0002702	medial amygdaloid nucleus	semapv:LexicalMatching	0.7
UBERON:0002894	olfactory cortex	skos:exactMatch	BTO:0001446	olfactory cortex	semapv:LexicalMatching	0.7
UBERON:0002925	trigeminal nucleus	skos:exactMatch	BTO:0001074	trigeminal nucleus	semapv:LexicalMatching	0.7
UBERON:0003050	olfactory placode	skos:exactMatch	BTO:0006222	olfactory placode	semapv:LexicalMatching	0.7
UBERON:0003053	ventricular zone	skos:exactMatch	BTO:0003654	ventricular zone	semapv:LexicalMatching	0.7
UBERON:0003055	periderm	skos:exactMatch	BTO:0003377	periderm	semapv:LexicalMatching	0.7
UBERON:0003060	pronephric duct	skos:exactMatch	BTO:0005991	pronephric duct	semapv:LexicalMatching	0.7
UBERON:0003072	optic cup	skos:exactMatch	BTO:0005351	optic cup	semapv:LexicalMatching	0.7
UBERON:0003074	mesonephric duct	skos:exactMatch	BTO:0005990	mesonephric duct	semapv:LexicalMatching	0.7
UBERON:0003075	neural plate	skos:exactMatch	BTO:0001765	neural plate	semapv:LexicalMatching	0.7
UBERON:0003079	floor plate	skos:exactMatch	BTO:0001720	floor plate	semapv:LexicalMatching	0.7
UBERON:0003082	myotome	skos:exactMatch	BTO:0000742	myotome	semapv:LexicalMatching	0.7
UBERON:0003084	heart primordium	skos:exactMatch	BTO:0001887	heart primordium	semapv:LexicalMatching	0.7
UBERON:0003089	sclerotome	skos:exactMatch	BTO:0006255	sclerotome	semapv:LexicalMatching	0.7
UBERON:0003091	thyroid primordium	skos:exactMatch	BTO:0004709	thyroid primordium	semapv:LexicalMatching	0.7
UBERON:0003104	mesenchyme	skos:exactMatch	BTO:0001393	mesenchyme	semapv:LexicalMatching	0.7
UBERON:0003126	trachea	skos:exactMatch	BTO:0001388	trachea	semapv:LexicalMatching	0.7
UBERON:0003128	cranium	skos:exactMatch	BTO:0001295	cranium	semapv:LexicalMatching	0.7
UBERON:0003215	alveolus	skos:exactMatch	BTO:0000060	alveolus	semapv:LexicalMatching	0.7
UBERON:0003295	pharyngeal gland	skos:exactMatch	BTO:0004849	pharyngeal gland	semapv:LexicalMatching	0.7
UBERON:0003351	pharyngeal epithelium	skos:exactMatch	BTO:0005240	pharyngeal epithelium	semapv:LexicalMatching	0.7
UBERON:0003662	forelimb muscle	skos:exactMatch	BTO:0000479	forelimb muscle	semapv:LexicalMatching	0.7
UBERON:0003687	foramen magnum	skos:exactMatch	BTO:0006035	foramen magnum	semapv:LexicalMatching	0.7
UBERON:0003861	neural arch	skos:exactMatch	BTO:0001763	neural arch	semapv:LexicalMatching	0.7
UBERON:0003911	choroid plexus epithelium	skos:exactMatch	BTO:0006008	choroid plexus epithelium	semapv:LexicalMatching	0.7
UBERON:0003937	reproductive gland	skos:exactMatch	BTO:0006162	reproductive gland	semapv:LexicalMatching	0.7
UBERON:0004016	dermatome	skos:exactMatch	BTO:0006248	dermatome	semapv:LexicalMatching	0.7
UBERON:0004027	chorionic plate	skos:exactMatch	BTO:0002879	chorionic plate	semapv:LexicalMatching	0.7
UBERON:0004044	anterior visceral endoderm	skos:exactMatch	BTO:0003361	anterior visceral endoderm	semapv:LexicalMatching	0.7
UBERON:0004086	brain ventricle	skos:exactMatch	BTO:0001442	brain ventricle	semapv:LexicalMatching	0.7
UBERON:0004087	vena cava	skos:exactMatch	BTO:0001438	vena cava	semapv:LexicalMatching	0.7
UBERON:0004188	glomerular epithelium	skos:exactMatch	BTO:0001515	glomerular epithelium	semapv:LexicalMatching	0.7
UBERON:0004189	glomerular endothelium	skos:exactMatch	BTO:0004631	glomerular endothelium	semapv:LexicalMatching	0.7
UBERON:0004203	cortical collecting duct	skos:exactMatch	BTO:0002643	cortical collecting duct	semapv:LexicalMatching	0.7
UBERON:0004205	inner medullary collecting duct	skos:exactMatch	BTO:0004544	inner medullary collecting duct	semapv:LexicalMatching	0.7
UBERON:0004222	stomach smooth muscle	skos:exactMatch	BTO:0001818	stomach smooth muscle	semapv:LexicalMatching	0.7
UBERON:0004228	urinary bladder smooth muscle	skos:exactMatch	BTO:0001849	urinary bladder smooth muscle	semapv:LexicalMatching	0.7
UBERON:0004234	iris smooth muscle	skos:exactMatch	BTO:0000655	iris smooth muscle	semapv:LexicalMatching	0.7
UBERON:0004243	prostate gland smooth muscle	skos:exactMatch	BTO:0003159	prostate gland smooth muscle	semapv:LexicalMatching	0.7
UBERON:0004340	allantois	skos:exactMatch	BTO:0000474	allantois	semapv:LexicalMatching	0.7
UBERON:0004345	trophectoderm	skos:exactMatch	BTO:0001840	trophectoderm	semapv:LexicalMatching	0.7
UBERON:0004347	limb bud	skos:exactMatch	BTO:0001640	limb bud	semapv:LexicalMatching	0.7
UBERON:0004364	ectoplacental cone	skos:exactMatch	BTO:0001715	ectoplacental cone	semapv:LexicalMatching	0.7
UBERON:0004449	cerebral artery	skos:exactMatch	BTO:0005205	cerebral artery	semapv:LexicalMatching	0.7
UBERON:0004535	cardiovascular system	skos:exactMatch	BTO:0000088	cardiovascular system	semapv:LexicalMatching	0.7
UBERON:0004572	arterial system	skos:exactMatch	BTO:0004690	arterial system	semapv:LexicalMatching	0.7
UBERON:0004582	venous system	skos:exactMatch	BTO:0004692	venous system	semapv:LexicalMatching	0.7
UBERON:0004638	blood vessel endothelium	skos:exactMatch	BTO:0000766	blood vessel endothelium	semapv:LexicalMatching	0.7
UBERON:0004681	vestibular system	skos:exactMatch	BTO:0003381	vestibular system	semapv:LexicalMatching	0.7
UBERON:0004711	jugular vein	skos:exactMatch	BTO:0001744	jugular vein	semapv:LexicalMatching	0.7
UBERON:0004715	annulus fibrosus disci intervertebralis	skos:exactMatch	BTO:0003629	annulus fibrosus disci intervertebralis	semapv:LexicalMatching	0.7
UBERON:0004716	conceptus	skos:exactMatch	BTO:0003834	conceptus	semapv:LexicalMatching	0.7
UBERON:0004734	gastrula	skos:exactMatch	BTO:0001403	gastrula	semapv:LexicalMatching	0.7
UBERON:0004758	salt gland	skos:exactMatch	BTO:0001204	salt gland	semapv:LexicalMatching	0.7
UBERON:0004804	oviduct epithelium	skos:exactMatch	BTO:0002402	oviduct epithelium	semapv:LexicalMatching	0.7
UBERON:0004809	salivary gland epithelium	skos:exactMatch	BTO:0004421	salivary gland epithelium	semapv:LexicalMatching	0.7
UBERON:0004820	bile duct epithelium	skos:exactMatch	BTO:0000417	bile duct epithelium	semapv:LexicalMatching	0.7
UBERON:0004851	aorta endothelium	skos:exactMatch	BTO:0000394	aorta endothelium	semapv:LexicalMatching	0.7
UBERON:0004877	visceral endoderm	skos:exactMatch	BTO:0003487	visceral endoderm	semapv:LexicalMatching	0.7
UBERON:0004893	interalveolar septum	skos:exactMatch	BTO:0003149	interalveolar septum	semapv:LexicalMatching	0.7
UBERON:0004894	alveolar wall	skos:exactMatch	BTO:0002543	alveolar wall	semapv:LexicalMatching	0.7
UBERON:0005167	papillary duct	skos:exactMatch	BTO:0004540	papillary duct	semapv:LexicalMatching	0.7
UBERON:0005184	hair medulla	skos:exactMatch	BTO:0004671	hair medulla	semapv:LexicalMatching	0.7
UBERON:0005290	myelencephalon	skos:exactMatch	BTO:0000758	myelencephalon	semapv:LexicalMatching	0.7
UBERON:0005292	extraembryonic tissue	skos:exactMatch	BTO:0003360	extraembryonic tissue	semapv:LexicalMatching	0.7
UBERON:0005305	thyroid follicle	skos:exactMatch	BTO:0004708	thyroid follicle	semapv:LexicalMatching	0.7
UBERON:0005317	pulmonary artery endothelium	skos:exactMatch	BTO:0000137	pulmonary artery endothelium	semapv:LexicalMatching	0.7
UBERON:0005366	olfactory lobe	skos:exactMatch	BTO:0001362	olfactory lobe	semapv:LexicalMatching	0.7
UBERON:0005382	dorsal striatum	skos:exactMatch	BTO:0004701	dorsal striatum	semapv:LexicalMatching	0.7
UBERON:0005398	female reproductive gland	skos:exactMatch	BTO:0000254	female reproductive gland	semapv:LexicalMatching	0.7
UBERON:0005399	male reproductive gland	skos:exactMatch	BTO:0000080	male reproductive gland	semapv:LexicalMatching	0.7
UBERON:0005403	ventral striatum	skos:exactMatch	BTO:0004702	ventral striatum	semapv:LexicalMatching	0.7
UBERON:0005408	circumventricular organ	skos:exactMatch	BTO:0006267	circumventricular organ	semapv:LexicalMatching	0.7
UBERON:0005609	iliac artery	skos:exactMatch	BTO:0004665	iliac artery	semapv:LexicalMatching	0.7
UBERON:0005616	mesenteric artery	skos:exactMatch	BTO:0000779	mesenteric artery	semapv:LexicalMatching	0.7
UBERON:0005617	mesenteric vein	skos:exactMatch	BTO:0002781	mesenteric vein	semapv:LexicalMatching	0.7
UBERON:0005805	dorsal aorta	skos:exactMatch	BTO:0004673	dorsal aorta	semapv:LexicalMatching	0.7
UBERON:0005813	tubercle	skos:exactMatch	BTO:0002173	tubercle	semapv:LexicalMatching	0.7
UBERON:0006444	annulus fibrosus	skos:exactMatch	BTO:0003627	annulus fibrosus	semapv:LexicalMatching	0.7
UBERON:0006562	pharynx	skos:exactMatch	BTO:0001049	pharynx	semapv:LexicalMatching	0.7
UBERON:0006568	hypothalamic nucleus	skos:exactMatch	BTO:0002449	hypothalamic nucleus	semapv:LexicalMatching	0.7
UBERON:0006660	muscular coat	skos:exactMatch	BTO:0004838	muscular coat	semapv:LexicalMatching	0.7
UBERON:0006676	muscularis mucosa	skos:exactMatch	BTO:0004839	muscularis mucosa	semapv:LexicalMatching	0.7
UBERON:0006799	glandular epithelium	skos:exactMatch	BTO:0002991	glandular epithelium	semapv:LexicalMatching	0.7
UBERON:0006849	scapula	skos:exactMatch	BTO:0001218	scapula	semapv:LexicalMatching	0.7
UBERON:0006913	lip epithelium	skos:exactMatch	BTO:0004468	lip epithelium	semapv:LexicalMatching	0.7
UBERON:0006914	squamous epithelium	skos:exactMatch	BTO:0002072	squamous epithelium	semapv:LexicalMatching	0.7
UBERON:0006953	ejaculatory duct epithelium	skos:exactMatch	BTO:0002618	ejaculatory duct epithelium	semapv:LexicalMatching	0.7
UBERON:0006955	uterine epithelium	skos:exactMatch	BTO:0003082	uterine epithelium	semapv:LexicalMatching	0.7
UBERON:0007106	chorionic villus	skos:exactMatch	BTO:0001161	chorionic villus	semapv:LexicalMatching	0.7
UBERON:0007228	vestibular nucleus	skos:exactMatch	BTO:0004368	vestibular nucleus	semapv:LexicalMatching	0.7
UBERON:0007318	saphenous vein	skos:exactMatch	BTO:0001808	saphenous vein	semapv:LexicalMatching	0.7
UBERON:0007329	pancreatic duct	skos:exactMatch	BTO:0002362	pancreatic duct	semapv:LexicalMatching	0.7
UBERON:0007777	umbilical vein endothelium	skos:exactMatch	BTO:0001416	umbilical vein endothelium	semapv:LexicalMatching	0.7
UBERON:0007778	umbilical artery endothelium	skos:exactMatch	BTO:0000857	umbilical artery endothelium	semapv:LexicalMatching	0.7
UBERON:0007798	vascular system	skos:exactMatch	BTO:0001085	vascular system	semapv:LexicalMatching	0.7
UBERON:0007806	connecting stalk	skos:exactMatch	BTO:0004705	connecting stalk	semapv:LexicalMatching	0.7
UBERON:0008266	periodontal ligament	skos:exactMatch	BTO:0001020	periodontal ligament	semapv:LexicalMatching	0.7
UBERON:0008281	tooth bud	skos:exactMatch	BTO:0001838	tooth bud	semapv:LexicalMatching	0.7
UBERON:0008304	inner chondrogenic layer of perichondrium	skos:exactMatch	BTO:0005090	inner chondrogenic layer of perichondrium	semapv:LexicalMatching	0.7
UBERON:0008305	outer fibrous layer of perichondrium	skos:exactMatch	BTO:0005091	outer fibrous layer of perichondrium	semapv:LexicalMatching	0.7
UBERON:0008307	heart endothelium	skos:exactMatch	BTO:0004293	heart endothelium	semapv:LexicalMatching	0.7
UBERON:0008331	clitoral smooth muscle	skos:exactMatch	BTO:0005000	clitoral smooth muscle	semapv:LexicalMatching	0.7
UBERON:0008339	microvascular endothelium	skos:exactMatch	BTO:0003122	microvascular endothelium	semapv:LexicalMatching	0.7
UBERON:0008367	breast epithelium	skos:exactMatch	BTO:0001428	breast epithelium	semapv:LexicalMatching	0.7
UBERON:0008397	tracheobronchial epithelium	skos:exactMatch	BTO:0002925	tracheobronchial epithelium	semapv:LexicalMatching	0.7
UBERON:0008408	distal tubular epithelium	skos:exactMatch	BTO:0003948	distal tubular epithelium	semapv:LexicalMatching	0.7
UBERON:0008836	liver bud	skos:exactMatch	BTO:0001642	liver bud	semapv:LexicalMatching	0.7
UBERON:0008904	neuromast	skos:exactMatch	BTO:0006219	neuromast	semapv:LexicalMatching	0.7
UBERON:0008969	dental follicle	skos:exactMatch	BTO:0000337	dental follicle	semapv:LexicalMatching	0.7
UBERON:0008974	apocrine gland	skos:exactMatch	BTO:0001162	apocrine gland	semapv:LexicalMatching	0.7
UBERON:0008987	renal parenchyma	skos:exactMatch	BTO:0003604	renal parenchyma	semapv:LexicalMatching	0.7
UBERON:0009010	periurethral tissue	skos:exactMatch	BTO:0005081	periurethral tissue	semapv:LexicalMatching	0.7
UBERON:0009127	epibranchial ganglion	skos:exactMatch	BTO:0006250	epibranchial ganglion	semapv:LexicalMatching	0.7
UBERON:0009758	abdominal ganglion	skos:exactMatch	BTO:0000022	abdominal ganglion	semapv:LexicalMatching	0.7
UBERON:0009773	renal tubule	skos:exactMatch	BTO:0000343	renal tubule	semapv:LexicalMatching	0.7
UBERON:0010143	seminal vesicle fluid	skos:exactMatch	BTO:0002053	seminal vesicle fluid	semapv:LexicalMatching	0.7
UBERON:0010152	skin mucus	skos:exactMatch	BTO:0005082	skin mucus	semapv:LexicalMatching	0.7
UBERON:0010210	blood clot	skos:exactMatch	BTO:0000102	blood clot	semapv:LexicalMatching	0.7
UBERON:0010243	merocrine gland	skos:exactMatch	BTO:0002324	merocrine gland	semapv:LexicalMatching	0.7
UBERON:0010302	amnioserosa	skos:exactMatch	BTO:0004800	amnioserosa	semapv:LexicalMatching	0.7
UBERON:0010754	germinal center	skos:exactMatch	BTO:0003671	germinal center	semapv:LexicalMatching	0.7
UBERON:0011004	pharyngeal arch cartilage	skos:exactMatch	BTO:0006051	pharyngeal arch cartilage	semapv:LexicalMatching	0.7
UBERON:0011148	submucosal gland	skos:exactMatch	BTO:0003751	submucosal gland	semapv:LexicalMatching	0.7
UBERON:0011374	prepuce	skos:exactMatch	BTO:0001113	prepuce	semapv:LexicalMatching	0.7
UBERON:0011997	coelom	skos:exactMatch	BTO:0001707	coelom	semapv:LexicalMatching	0.7
UBERON:0012168	umbilical cord blood	skos:exactMatch	BTO:0004053	umbilical cord blood	semapv:LexicalMatching	0.7
UBERON:0012248	cervical mucosa	skos:exactMatch	BTO:0000411	cervical mucosa	semapv:LexicalMatching	0.7
UBERON:0012344	holocrine gland	skos:exactMatch	BTO:0002325	holocrine gland	semapv:LexicalMatching	0.7
UBERON:0012615	umbilical smooth muscle	skos:exactMatch	BTO:0001847	umbilical smooth muscle	semapv:LexicalMatching	0.7
UBERON:0013479	lung endothelium	skos:exactMatch	BTO:0004128	lung endothelium	semapv:LexicalMatching	0.7
UBERON:0013694	brain endothelium	skos:exactMatch	BTO:0003248	brain endothelium	semapv:LexicalMatching	0.7
UBERON:0013755	arterial blood	skos:exactMatch	BTO:0006188	arterial blood	semapv:LexicalMatching	0.7
UBERON:0013756	venous blood	skos:exactMatch	BTO:0006187	venous blood	semapv:LexicalMatching	0.7
UBERON:0016525	frontal lobe	skos:exactMatch	BTO:0000484	frontal lobe	semapv:LexicalMatching	0.7
UBERON:0017654	buccal gland	skos:exactMatch	BTO:0005767	buccal gland	semapv:LexicalMatching	0.7
UBERON:0019143	intramuscular adipose tissue	skos:exactMatch	BTO:0004043	intramuscular adipose tissue	semapv:LexicalMatching	0.7
UBERON:0019189	carotid artery endothelium	skos:exactMatch	BTO:0004626	carotid artery endothelium	semapv:LexicalMatching	0.7
UBERON:0019196	iliac artery endothelium	skos:exactMatch	BTO:0004661	iliac artery endothelium	semapv:LexicalMatching	0.7
UBERON:0019204	skin epithelium	skos:exactMatch	BTO:0004382	skin epithelium	semapv:LexicalMatching	0.7
UBERON:0035050	excretory duct	skos:exactMatch	BTO:0006341	excretory duct	semapv:LexicalMatching	0.7
UBERON:0036217	coelomic fluid	skos:exactMatch	BTO:0001708	coelomic fluid	semapv:LexicalMatching	0.7
UBERON:0036243	vaginal fluid	skos:exactMatch	BTO:0003084	vaginal fluid	semapv:LexicalMatching	0.7
UBERON:8410037	high endothelial venule	skos:exactMatch	BTO:0004528	high endothelial venule	semapv:LexicalMatching	0.7
UBERON:8450002	excretory system	skos:exactMatch	BTO:0006338	excretory system	semapv:LexicalMatching	0.7
UBERON:8600038	placental disc	skos:exactMatch	BTO:0003678	placental disc	semapv:LexicalMatching	0.7
```

## 4. Merge KBs

We use the Python API to merge because the label-match candidates are already
in memory as `PFact` objects. The `KB.extend()` method produces a new KB with
facts, pfacts, and labels from both sources plus our mapping candidates.

In [6]:
from boomer.io import save_kb

merged = cl_kb.extend(
    facts=bto_kb.facts,
    pfacts=bto_kb.pfacts + label_pfacts,
    labels={**bto_kb.labels},
)
merged.name = "CL-BTO Alignment"
merged.normalize()

print(f"Merged KB: {len(merged.facts)} hard facts, {len(merged.pfacts)} pfacts, {len(merged.labels)} labels")

merged_path = DIR / "merged.yaml"
save_kb(merged, merged_path)
print(f"Saved merged KB to {merged_path}")

Merged KB: 90474 hard facts, 32861 pfacts, 25317 labels


Saved merged KB to docs/tutorial/real-world-alignment-files/merged.yaml


## 5. Extract Module Around Epithelial Cell

The full merged KB is too large to solve in reasonable time. We extract a
local neighborhood around `CL:0000066` (epithelial cell) using `--max-hops 1`.
We also use `-C 20` (max pfacts per clique) to ensure the solver partitions
effectively and completes quickly.

In [7]:
%%bash
DIR=docs/tutorial/real-world-alignment-files

uv run python -m boomer.cli extract \
    $DIR/merged.yaml \
    --id CL:0000066 \
    --max-hops 1 \
    -o $DIR/epithelial_cluster.yaml

Extracted sub-KB from docs/tutorial/real-world-alignment-files/merged.yaml to docs/tutorial/real-wor

ld-alignment-files/epithelial_cluster.yaml (yaml)
Original KB: 90474 facts, 32861 pfacts
Extracted K

B: 345 facts, 50 pfacts
Seeds: ['CL:0000066']


In [8]:
from boomer.io import load_kb

cluster = load_kb(DIR / "epithelial_cluster.yaml")
print(f"Cluster: {len(cluster.facts)} hard facts, {len(cluster.pfacts)} pfacts, {len(cluster.labels)} labels")
print(f"Full KB: {len(merged.facts)} hard facts, {len(merged.pfacts)} pfacts, {len(merged.labels)} labels")

print(f"\nEntities in cluster ({len(cluster.labels)} labeled):")
for eid, label in sorted(cluster.labels.items()):
    print(f"  {eid}: {label}")

Cluster: 345 hard facts, 50 pfacts, 44 labels
Full KB: 90474 hard facts, 32861 pfacts, 25317 labels

Entities in cluster (44 labeled):
  BTO:0000414: epithelial cell
  CL:0000066: epithelial cell
  CL:0000067: ciliated epithelial cell
  CL:0000068: duct epithelial cell
  CL:0000075: columnar/cuboidal epithelial cell
  CL:0000076: squamous epithelial cell
  CL:0000079: stratified epithelial cell
  CL:0000098: sensory epithelial cell
  CL:0000182: hepatocyte
  CL:0000244: transitional epithelial cell
  CL:0000255: eukaryotic cell
  CL:0000411: Caenorhabditis hypodermal cell
  CL:0000418: arcade cell
  CL:0000420: syncytial epithelial cell
  CL:0000500: follicular epithelial cell
  CL:0000658: cuticle secreting cell
  CL:0000730: leading edge cell
  CL:0000846: vestibular dark cell
  CL:0002076: endo-epithelial cell
  CL:0002077: ecto-epithelial cell
  CL:0002078: meso-epithelial cell
  CL:0002204: tuft cell
  CL:0002222: vertebrate lens cell
  CL:0002327: mammary gland epithelial cell
  

## 6. Solve

Run BOOMER's probabilistic search on the extracted cluster. The `-C 20` flag
tells the solver to partition cliques with more than 20 pfacts, keeping each
sub-problem tractable. The solver evaluates all combinations of truth
assignments, constrained by logical consistency (e.g., `MemberOfDisjointGroup`
prevents cross-prefix equivalence conflicts).

In [9]:
%%bash
DIR=docs/tutorial/real-world-alignment-files

uv run python -m boomer.cli solve \
    $DIR/epithelial_cluster.yaml \
    --timeout 120 \
    -C 20 \
    -O yaml \
    -o $DIR/solution.yaml \
    --quiet

Solving KB: CL-BTO Alignment with 50 pfacts; threshold=200
Partitioning KB, num pfacts= 50 sub-KBs (

threshold=200)
Partitioning KB into 265 sub-KBs
Solving sub-KB: 92 facts, 1 pfacts
Solving KB: CL-BT

O Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving su

b-KB: 59 facts, 8 pfacts
Solving KB: CL-BTO Alignment with 8 pfacts; threshold=8
Sub-solution: pr:0.

05764800999999997 // post:0.036953852564102795 // 258 combinations
Solving sub-KB: 7 facts, 2 pfacts

...

binations
Solving sub-KB: 2 facts, 1 pfacts
Solving KB: CL-BTO Alignment with 1 pfacts; threshold=1


Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving sub-KB: 2 facts, 1 pfacts
Solving KB: CL-

BTO Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving 

sub-KB: 3 facts, 1 pfacts
Solving KB: CL-BTO Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0

.7 // post:0.7 // 2 combinations


## 7. Interpret Results

Let's load the solution and examine which EquivalentTo mappings were
accepted vs rejected, and how prior probabilities shifted.

In [10]:
import yaml

from boomer.model import Solution

sol_data = yaml.safe_load((DIR / "solution.yaml").read_text())
solution = Solution.model_validate(sol_data)

print(f"Confidence: {solution.confidence:.4f}")
print(f"Prior prob: {solution.prior_prob:.6f}")
print(f"Posterior prob: {solution.posterior_prob:.6f}")
if solution.timed_out:
    print("(search timed out — results are approximate)")

print("\n=== EquivalentTo Mappings ===")
print(f"{'Status':<10} {'Prior':>6} {'Post':>6}  Subject -> Object")
print("-" * 70)

equiv_facts = [
    sp for sp in solution.solved_pfacts
    if sp.pfact.fact.fact_type == "EquivalentTo"
]
for sp in sorted(equiv_facts, key=lambda s: s.posterior_prob, reverse=True):
    f = sp.pfact.fact
    status = "ACCEPT" if sp.truth_value else "REJECT"
    sub_label = cluster.labels.get(f.sub, "")
    obj_label = cluster.labels.get(f.equivalent, "")
    print(
        f"{status:<10} {sp.pfact.prob:>5.2f} -> {sp.posterior_prob:>5.2f}  "
        f"{f.sub} ({sub_label}) == {f.equivalent} ({obj_label})"
    )

Confidence: 0.0000
Prior prob: 0.000000
Posterior prob: 0.000000

=== EquivalentTo Mappings ===
Status      Prior   Post  Subject -> Object
----------------------------------------------------------------------
ACCEPT      0.70 ->  0.94  CL:0000066 (epithelial cell) == BTO:0000414 (epithelial cell)
ACCEPT      0.70 ->  0.94  CL:0000066 (epithelial cell) == BTO:0000414 (epithelial cell)
ACCEPT      0.70 ->  0.94  CL:0000182 (hepatocyte) == BTO:0000575 ()
ACCEPT      0.70 ->  0.94  CL:0000182 (hepatocyte) == BTO:0000575 ()
ACCEPT      0.70 ->  0.70  CL:0000066 (epithelial cell) == CALOHA:TS-2026 ()
ACCEPT      0.70 ->  0.70  CL:0000066 (epithelial cell) == CARO:0000077 ()
ACCEPT      0.70 ->  0.70  CL:0000066 (epithelial cell) == FBbt:00000124 ()
ACCEPT      0.70 ->  0.70  CL:0000066 (epithelial cell) == FMA:66768 ()
ACCEPT      0.70 ->  0.70  CL:0000066 (epithelial cell) == WBbt:0003672 ()
ACCEPT      0.70 ->  0.70  CL:0000066 (epithelial cell) == ZFA:0009034 ()
ACCEPT      0.70 ->  0.7

## 8. Export as SSSOM and OBOGraphs

BOOMER can export solutions in standard ontology formats:
- **SSSOM TSV** — the standard for ontology mappings, consumable by sssom-py / OAK
- **OBOGraphs JSON** — the standard graph exchange format for OBO ontologies

In [11]:
%%bash
DIR=docs/tutorial/real-world-alignment-files

uv run python -m boomer.cli solve \
    $DIR/epithelial_cluster.yaml \
    --timeout 120 \
    -C 20 \
    -O sssom \
    -o $DIR/solution.sssom.tsv \
    --quiet

Solving KB: CL-BTO Alignment with 50 pfacts; threshold=200
Partitioning KB, num pfacts= 50 sub-KBs (

threshold=200)
Partitioning KB into 265 sub-KBs
Solving sub-KB: 92 facts, 1 pfacts
Solving KB: CL-BT

O Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving su

b-KB: 59 facts, 8 pfacts
Solving KB: CL-BTO Alignment with 8 pfacts; threshold=8
Sub-solution: pr:0.

05764800999999997 // post:0.036953852564102795 // 258 combinations
Solving sub-KB: 7 facts, 2 pfacts

...

binations
Solving sub-KB: 2 facts, 1 pfacts
Solving KB: CL-BTO Alignment with 1 pfacts; threshold=1


Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving sub-KB: 2 facts, 1 pfacts
Solving KB: CL-

BTO Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving 

sub-KB: 3 facts, 1 pfacts
Solving KB: CL-BTO Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0

.7 // post:0.7 // 2 combinations


In [12]:
show(DIR / "solution.sssom.tsv", "tsv")

**`solution.sssom.tsv`**

```tsv
#mapping_set_id: boomer:solution
#mapping_tool: BOOMER
#mapping_date: 2026-03-04
#curie_map:
#  BTO: http://purl.obolibrary.org/obo/BTO_
#  CL: http://purl.obolibrary.org/obo/CL_
#  FBbt: http://purl.obolibrary.org/obo/FBbt_
#  WBbt: http://purl.obolibrary.org/obo/WBbt_
#  ZFA: http://purl.obolibrary.org/obo/ZFA_
#  semapv: https://w3id.org/semapv/vocab/
#  skos: http://www.w3.org/2004/02/skos/core#
subject_id	subject_label	predicate_id	object_id	object_label	mapping_justification	confidence
CL:0000255	eukaryotic cell	skos:exactMatch	MESH:D005057		semapv:CompositeMatching	0.7
CL:0000066	epithelial cell	skos:exactMatch	BTO:0000414	epithelial cell	semapv:CompositeMatching	0.9423076923076933
CL:0000066	epithelial cell	skos:exactMatch	CALOHA:TS-2026		semapv:CompositeMatching	0.7000000000000036
CL:0000066	epithelial cell	skos:exactMatch	CARO:0000077		semapv:CompositeMatching	0.7000000000000036
CL:0000066	epithelial cell	skos:exactMatch	FBbt:00000124		semapv:CompositeMatching	0.7000000000000036
CL:0000066	epithelial cell	skos:exactMatch	FMA:66768		semapv:CompositeMatching	0.7000000000000036
CL:0000066	epithelial cell	skos:exactMatch	WBbt:0003672		semapv:CompositeMatching	0.7000000000000036
CL:0000066	epithelial cell	skos:exactMatch	ZFA:0009034		semapv:CompositeMatching	0.7000000000000036
CL:0000066	epithelial cell	skos:exactMatch	BTO:0000414	epithelial cell	semapv:CompositeMatching	0.9423076923076933
CL:0002077	ecto-epithelial cell	skos:exactMatch	FMA:69074		semapv:CompositeMatching	0.7
CL:0002077	ecto-epithelial cell	skos:exactMatch	ZFA:0009385		semapv:CompositeMatching	0.7
CL:0000067	ciliated epithelial cell	skos:exactMatch	FMA:70605		semapv:CompositeMatching	0.7
CL:0000067	ciliated epithelial cell	skos:exactMatch	ZFA:0009035		semapv:CompositeMatching	0.7
CL:0000068	duct epithelial cell	skos:exactMatch	ZFA:0009372		semapv:CompositeMatching	0.7
CL:0002078	meso-epithelial cell	skos:exactMatch	FMA:69076		semapv:CompositeMatching	0.7
CL:0002078	meso-epithelial cell	skos:exactMatch	ZFA:0009388		semapv:CompositeMatching	0.7
CL:0000075	columnar/cuboidal epithelial cell	skos:exactMatch	ZFA:0009038		semapv:CompositeMatching	0.7
CL:0000076	squamous epithelial cell	skos:exactMatch	CALOHA:TS-1249		semapv:CompositeMatching	0.7
CL:0000076	squamous epithelial cell	skos:exactMatch	ZFA:0009039		semapv:CompositeMatching	0.7
CL:0000079	stratified epithelial cell	skos:exactMatch	ZFA:0009042		semapv:CompositeMatching	0.7
CL:0002076	endo-epithelial cell	skos:exactMatch	FMA:69075		semapv:CompositeMatching	0.7
CL:0002076	endo-epithelial cell	skos:exactMatch	ZFA:0009383		semapv:CompositeMatching	0.7
CL:0000098	sensory epithelial cell	skos:exactMatch	BTO:0004301		semapv:CompositeMatching	0.7
CL:0000098	sensory epithelial cell	skos:exactMatch	ZFA:0009050		semapv:CompositeMatching	0.7
CL:0000182	hepatocyte	skos:exactMatch	BTO:0000575		semapv:CompositeMatching	0.9423076923076922
CL:0000182	hepatocyte	skos:exactMatch	CALOHA:TS-0454		semapv:CompositeMatching	0.7
CL:0000182	hepatocyte	skos:exactMatch	FMA:14515		semapv:CompositeMatching	0.7
CL:0000182	hepatocyte	skos:exactMatch	ZFA:0009111		semapv:CompositeMatching	0.7
CL:0000182	hepatocyte	skos:exactMatch	BTO:0000575		semapv:CompositeMatching	0.9423076923076922
CL:0000244	transitional epithelial cell	skos:exactMatch	FMA:66778		semapv:CompositeMatching	0.7
CL:0000244	transitional epithelial cell	skos:exactMatch	ZFA:0009148		semapv:CompositeMatching	0.7
CL:0000411	Caenorhabditis hypodermal cell	skos:exactMatch	WBbt:0007846		semapv:CompositeMatching	0.7
CL:0000411	Caenorhabditis hypodermal cell	skos:exactMatch	ZFA:0009196		semapv:CompositeMatching	0.7
CL:0000418	arcade cell	skos:exactMatch	WBbt:0005793		semapv:CompositeMatching	0.7
CL:0000420	syncytial epithelial cell	skos:exactMatch	Wikipedia:Syncytium		semapv:CompositeMatching	0.7
CL:0002204	tuft cell	skos:exactMatch	FMA:67978		semapv:CompositeMatching	0.7
CL:0002222	vertebrate lens cell	skos:exactMatch	FMA:70950		semapv:CompositeMatching	0.7
CL:0002327	mammary gland epithelial cell	skos:exactMatch	BTO:0004300		semapv:CompositeMatching	0.7
CL:0002518	kidney epithelial cell	skos:exactMatch	KUPO:0001019		semapv:CompositeMatching	0.7
CL:0002518	kidney epithelial cell	skos:exactMatch	ZFA:0009374		semapv:CompositeMatching	0.7
CL:0002586	retinal pigment epithelial cell	skos:exactMatch	BTO:0004910		semapv:CompositeMatching	0.7
CL:0002586	retinal pigment epithelial cell	skos:exactMatch	FMA:75802		semapv:CompositeMatching	0.7
CL:0005006	ionocyte	skos:exactMatch	ZFA:0005323		semapv:CompositeMatching	0.7
CL:0008026	open tracheal system tracheocyte	skos:exactMatch	FBbt:00005038		semapv:CompositeMatching	0.7
CL:1000296	epithelial cell of urethra	skos:exactMatch	FMA:256165		semapv:CompositeMatching	0.7
CL:1000415	epithelial cell of gallbladder	skos:exactMatch	FMA:67780		semapv:CompositeMatching	0.7
CL:1000432	conjunctival epithelial cell	skos:exactMatch	FMA:70552		semapv:CompositeMatching	0.7
CL:1000433	epithelial cell of lacrimal canaliculus	skos:exactMatch	FMA:70553		semapv:CompositeMatching	0.7
CL:1000434	epithelial cell of external acoustic meatus	skos:exactMatch	FMA:70555		semapv:CompositeMatching	0.7
CL:1000441	epithelial cell of viscerocranial mucosa	skos:exactMatch	FMA:70583		semapv:CompositeMatching	0.7
```

In [13]:
%%bash
DIR=docs/tutorial/real-world-alignment-files

uv run python -m boomer.cli solve \
    $DIR/epithelial_cluster.yaml \
    --timeout 120 \
    -C 20 \
    -O obographs \
    -o $DIR/solution.obographs.json \
    --quiet

Solving KB: CL-BTO Alignment with 50 pfacts; threshold=200
Partitioning KB, num pfacts= 50 sub-KBs (

threshold=200)
Partitioning KB into 265 sub-KBs
Solving sub-KB: 92 facts, 1 pfacts
Solving KB: CL-BT

O Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving su

b-KB: 59 facts, 8 pfacts
Solving KB: CL-BTO Alignment with 8 pfacts; threshold=8
Sub-solution: pr:0.

05764800999999997 // post:0.036953852564102795 // 258 combinations
Solving sub-KB: 7 facts, 2 pfacts

...

binations
Solving sub-KB: 2 facts, 1 pfacts
Solving KB: CL-BTO Alignment with 1 pfacts; threshold=1


Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving sub-KB: 2 facts, 1 pfacts
Solving KB: CL-

BTO Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0.7 // post:0.7 // 2 combinations
Solving 

sub-KB: 3 facts, 1 pfacts
Solving KB: CL-BTO Alignment with 1 pfacts; threshold=1
Sub-solution: pr:0

.7 // post:0.7 // 2 combinations


In [14]:
show(DIR / "solution.obographs.json", "json")

**`solution.obographs.json`**

```json
{
  "graphs": [
    {
      "id": "boomer:solution",
      "meta": {
        "basicPropertyValues": [
          {
            "pred": "boomer:confidence",
            "val": "1.642809059088352e-05"
          },
          {
            "pred": "boomer:combinations",
            "val": "366"
          }
        ]
      },
      "nodes": [
        {
          "id": "BFO:0000002"
        },
        {
          "id": "BTO:0000414",
          "lbl": "epithelial cell"
        },
        {
          "id": "BTO:0000575"
        },
        {
          "id": "BTO:0001514"
        },
        {
          "id": "BTO:0001540"
        },
        {
          "id": "BTO:0001663"
        },
        {
          "id": "BTO:0002295"
        },
        {
          "id": "BTO:0002309"
        },
        {
          "id": "BTO:0002484"
        },
        {
          "id": "BTO:0003736"
        },
        {
          "id": "BTO:0004300"
        },
        {
          "id": "BTO:0004301"
        },
        {
          "id": "BTO:0004910"
        },
        {
          "id": "CALOHA:TS-0454"
        },
        {
          "id": "CALOHA:TS-1249"
        },
        {
          "id": "CALOHA:TS-2026"
        },
        {
          "id": "CARO:0000077"
        },
        {
          "id": "CL:0000000"
        },
        {
          "id": "CL:0000018"
        },
        {
          "id": "CL:0000019"
        },
        {
          "id": "CL:0000021"
        },
        {
          "id": "CL:0000026"
        },
        {
          "id": "CL:0000032"
        },
        {
          "id": "CL:0000034"
        },
        {
          "id": "CL:0000059"
        },
        {
          "id": "CL:0000064"
        },
        {
          "id": "CL:0000065"
        },
        {
          "id": "CL:0000066",
          "lbl": "epithelial cell"
        },
        {
          "id": "CL:0000067",
          "lbl": "ciliated epithelial cell"
        },
        {
          "id": "CL:0000068",
          "lbl": "duct epithelial cell"
        },
        {
          "id": "CL:0000069"
        },
        {
          "id": "CL:0000072"
        },
        {
          "id": "CL:0000075",
          "lbl": "columnar/cuboidal epithelial cell"
        },
        {
          "id": "CL:0000076",
          "lbl": "squamous epithelial cell"
        },
        {
          "id": "CL:0000077"
        },
        {
          "id": "CL:0000078"
        },
        {
          "id": "CL:0000079",
          "lbl": "stratified epithelial cell"
        },
        {
          "id": "CL:0000080"
        },
        {
          "id": "CL:0000083"
        },
        {
          "id": "CL:0000098",
          "lbl": "sensory epithelial cell"
        },
        {
          "id": "CL:0000115"
        },
        {
          "id": "CL:0000146"
        },
        {
          "id": "CL:0000148"
        },
        {
          "id": "CL:0000149"
        },
        {
          "id": "CL:0000150"
        },
        {
          "id": "CL:0000151"
        },
        {
          "id": "CL:0000152"
        },
        {
          "id": "CL:0000160"
        },
        {
          "id": "CL:0000163"
        },
        {
          "id": "CL:0000164"
        },
        {
          "id": "CL:0000165"
        },
        {
          "id": "CL:0000168"
        },
        {
          "id": "CL:0000179"
        },
        {
          "id": "CL:0000182",
          "lbl": "hepatocyte"
        },
        {
          "id": "CL:0000185"
        },
        {
          "id": "CL:0000187"
        },
        {
          "id": "CL:0000188"
        },
        {
          "id": "CL:0000197"
        },
        {
          "id": "CL:0000209"
        },
        {
          "id": "CL:0000214"
        },
        {
          "id": "CL:0000216"
        },
        {
          "id": "CL:0000221"
        },
        {
          "id": "CL:0000222"
        },
        {
          "id": "CL:0000223"
        },
        {
          "id": "CL:0000226"
        },
        {
          "id": "CL:0000228"
        },
        {
          "id": "CL:0000237"
        },
        {
          "id": "CL:0000239"
        },
        {
          "id": "CL:0000240"
        },
        {
          "id": "CL:0000241"
        },
        {
          "id": "CL:0000242"
        },
        {
          "id": "CL:0000244",
          "lbl": "transitional epithelial cell"
        },
        {
          "id": "CL:0000255",
          "lbl": "eukaryotic cell"
        },
        {
          "id": "CL:0000257"
        },
        {
          "id": "CL:0000306"
        },
        {
          "id": "CL:0000327"
        },
        {
          "id": "CL:0000342"
        },
        {
          "id": "CL:0000345"
        },
        {
          "id": "CL:0000347"
        },
        {
          "id": "CL:0000348"
        },
        {
          "id": "CL:0000349"
        },
        {
          "id": "CL:0000360"
        },
        {
          "id": "CL:0000361"
        },
        {
          "id": "CL:0000365"
        },
        {
          "id": "CL:0000382"
        },
        {
          "id": "CL:0000387"
        },
        {
          "id": "CL:0000389"
        },
        {
          "id": "CL:0000407"
        },
        {
          "id": "CL:0000411",
          "lbl": "Caenorhabditis hypodermal cell"
        },
        {
          "id": "CL:0000417"
        },
        {
          "id": "CL:0000418",
          "lbl": "arcade cell"
        },
        {
          "id": "CL:0000420",
          "lbl": "syncytial epithelial cell"
        },
        {
          "id": "CL:0000425"
        },
        {
          "id": "CL:0000430"
        },
        {
          "id": "CL:0000431"
        },
        {
          "id": "CL:0000434"
        },
        {
          "id": "CL:0000436"
        },
        {
          "id": "CL:0000477"
        },
        {
          "id": "CL:0000487"
        },
        {
          "id": "CL:0000500",
          "lbl": "follicular epithelial cell"
        },
        {
          "id": "CL:0000517"
        },
        {
          "id": "CL:0000518"
        },
        {
          "id": "CL:0000520"
        },
        {
          "id": "CL:0000521"
        },
        {
          "id": "CL:0000571"
        },
        {
          "id": "CL:0000574"
        },
        {
          "id": "CL:0000575"
        },
        {
          "id": "CL:0000618"
        },
        {
          "id": "CL:0000656"
        },
        {
          "id": "CL:0000657"
        },
        {
          "id": "CL:0000658",
          "lbl": "cuticle secreting cell"
        },
        {
          "id": "CL:0000659"
        },
        {
          "id": "CL:0000677"
        },
        {
          "id": "CL:0000686"
        },
        {
          "id": "CL:0000710"
        },
        {
          "id": "CL:0000722"
        },
        {
          "id": "CL:0000730",
          "lbl": "leading edge cell"
        },
        {
          "id": "CL:0000731"
        },
        {
          "id": "CL:0000738"
        },
        {
          "id": "CL:0000747"
        },
        {
          "id": "CL:0000763"
        },
        {
          "id": "CL:0000837"
        },
        {
          "id": "CL:0000846",
          "lbl": "vestibular dark cell"
        },
        {
          "id": "CL:0000851"
        },
        {
          "id": "CL:0000852"
        },
        {
          "id": "CL:0000892"
        },
        {
          "id": "CL:0001035"
        },
        {
          "id": "CL:0002031"
        },
        {
          "id": "CL:0002032"
        },
        {
          "id": "CL:0002062"
        },
        {
          "id": "CL:0002075"
        },
        {
          "id": "CL:0002076",
          "lbl": "endo-epithelial cell"
        },
        {
          "id": "CL:0002077",
          "lbl": "ecto-epithelial cell"
        },
        {
          "id": "CL:0002078",
          "lbl": "meso-epithelial cell"
        },
        {
          "id": "CL:0002097"
        },
        {
          "id": "CL:0002149"
        },
        {
          "id": "CL:0002153"
        },
        {
          "id": "CL:0002157"
        },
        {
          "id": "CL:0002159"
        },
        {
          "id": "CL:0002162"
        },
        {
          "id": "CL:0002167"
        },
        {
          "id": "CL:0002174"
        },
        {
          "id": "CL:0002179"
        },
        {
          "id": "CL:0002180"
        },
        {
          "id": "CL:0002181"
        },
        {
          "id": "CL:0002198"
        },
        {
          "id": "CL:0002204",
          "lbl": "tuft cell"
        },
        {
          "id": "CL:0002222",
          "lbl": "vertebrate lens cell"
        },
        {
          "id": "CL:0002224"
        },
        {
          "id": "CL:0002231"
        },
        {
          "id": "CL:0002236"
        },
        {
          "id": "CL:0002237"
        },
        {
          "id": "CL:0002239"
        },
        {
          "id": "CL:0002249"
        },
        {
          "id": "CL:0002251"
        },
        {
          "id": "CL:0002257"
        },
        {
          "id": "CL:0002260"
        },
        {
          "id": "CL:0002261"
        },
        {
          "id": "CL:0002293"
        },
        {
          "id": "CL:0002305"
        },
        {
          "id": "CL:0002306"
        },
        {
          "id": "CL:0002318"
        },
        {
          "id": "CL:0002319"
        },
        {
          "id": "CL:0002320"
        },
        {
          "id": "CL:0002324"
        },
        {
          "id": "CL:0002326"
        },
        {
          "id": "CL:0002327",
          "lbl": "mammary gland epithelial cell"
        },
        {
          "id": "CL:0002368"
        },
        {
          "id": "CL:0002491"
        },
        {
          "id": "CL:0002494"
        },
        {
          "id": "CL:0002518",
          "lbl": "kidney epithelial cell"
        },
        {
          "id": "CL:0002519"
        },
        {
          "id": "CL:0002522"
        },
        {
          "id": "CL:0002536",
          "lbl": "epithelial cell of amnion"
        },
        {
          "id": "CL:0002559"
        },
        {
          "id": "CL:0002565",
          "lbl": "iris pigment epithelial cell"
        },
        {
          "id": "CL:0002577",
          "lbl": "placental epithelial cell"
        },
        {
          "id": "CL:0002584"
        },
        {
          "id": "CL:0002586",
          "lbl": "retinal pigment epithelial cell"
        },
        {
          "id": "CL:0002625"
        },
        {
          "id": "CL:0002665"
        },
        {
          "id": "CL:0005006",
          "lbl": "ionocyte"
        },
        {
          "id": "CL:0005009"
        },
        {
          "id": "CL:0005010"
        },
        {
          "id": "CL:0005012"
        },
        {
          "id": "CL:0005013"
        },
        {
          "id": "CL:0007005"
        },
        {
          "id": "CL:0007021"
        },
        {
          "id": "CL:0007023"
        },
        {
          "id": "CL:0008026",
          "lbl": "open tracheal system tracheocyte"
        },
        {
          "id": "CL:0008055"
        },
        {
          "id": "CL:0009003"
        },
        {
          "id": "CL:0009004"
        },
        {
          "id": "CL:0009005"
        },
        {
          "id": "CL:0009010"
        },
        {
          "id": "CL:0010002",
          "lbl": "epithelial cell of umbilical artery"
        },
        {
          "id": "CL:0010015",
          "lbl": "coronet cell"
        },
        {
          "id": "CL:0011004"
        },
        {
          "id": "CL:0011026"
        },
        {
          "id": "CL:0011029"
        },
        {
          "id": "CL:0017000"
        },
        {
          "id": "CL:0017003"
        },
        {
          "id": "CL:0017010"
        },
        {
          "id": "CL:0019001"
        },
        {
          "id": "CL:0019026"
        },
        {
          "id": "CL:0019028"
        },
        {
          "id": "CL:0019029"
        },
        {
          "id": "CL:0019032"
        },
        {
          "id": "CL:1000155"
        },
        {
          "id": "CL:1000182"
        },
        {
          "id": "CL:1000223"
        },
        {
          "id": "CL:1000272"
        },
        {
          "id": "CL:1000296",
          "lbl": "epithelial cell of urethra"
        },
        {
          "id": "CL:1000415",
          "lbl": "epithelial cell of gallbladder"
        },
        {
          "id": "CL:1000419"
        },
        {
          "id": "CL:1000432",
          "lbl": "conjunctival epithelial cell"
        },
        {
          "id": "CL:1000433",
          "lbl": "epithelial cell of lacrimal canaliculus"
        },
        {
          "id": "CL:1000434",
          "lbl": "epithelial cell of external acoustic meatus"
        },
        {
          "id": "CL:1000441",
          "lbl": "epithelial cell of viscerocranial mucosa"
        },
        {
          "id": "CL:1000449"
        },
        {
          "id": "CL:1000497"
        },
        {
          "id": "CL:1000600"
        },
        {
          "id": "CL:1000601"
        },
        {
          "id": "CL:1000703"
        },
        {
          "id": "CL:1000850"
        },
        {
          "id": "CL:1001320"
        },
        {
          "id": "CL:1001430"
        },
        {
          "id": "CL:1001575"
        },
        {
          "id": "CL:1001576"
        },
        {
          "id": "CL:1001577"
        },
        {
          "id": "CL:1001578"
        },
        {
          "id": "CL:1001586"
        },
        {
          "id": "CL:1001590"
        },
        {
          "id": "CL:1001591"
        },
        {
          "id": "CL:1001592"
        },
        {
          "id": "CL:1001597"
        },
        {
          "id": "CL:1001598"
        },
        {
          "id": "CL:1100001",
          "lbl": "secretory epithelial cell"
        },
        {
          "id": "CL:2000020"
        },
        {
          "id": "CL:2000021"
        },
        {
          "id": "CL:2000064"
        },
        {
          "id": "CL:2000084"
        },
        {
          "id": "CL:2000094"
        },
        {
          "id": "CL:4030007"
        },
        {
          "id": "CL:4030012"
        },
        {
          "id": "CL:4030023"
        },
        {
          "id": "CL:4030024",
          "lbl": "hillock cell"
        },
        {
          "id": "CL:4030031"
        },
        {
          "id": "CL:4030066"
        },
        {
          "id": "CL:4032000"
        },
        {
          "id": "CL:4033013"
        },
        {
          "id": "CL:4033023",
          "lbl": "airway submucosal gland collecting duct epithelial cell"
        },
        {
          "id": "CL:4033048"
        },
        {
          "id": "CL:4033055"
        },
        {
          "id": "CL:4033066"
        },
        {
          "id": "CL:4040006"
        },
        {
          "id": "CL:4042019"
        },
        {
          "id": "CL:4047056"
        },
        {
          "id": "CL:4052002"
        },
        {
          "id": "CL:4052018",
          "lbl": "fallopian tube epithelial cell"
        },
        {
          "id": "CL:4052019"
        },
        {
          "id": "CL:4052033"
        },
        {
          "id": "CL:4052034"
        },
        {
          "id": "CL:4052035"
        },
        {
          "id": "CL:4052036"
        },
        {
          "id": "CL:4052039"
        },
        {
          "id": "CL:4052040"
        },
        {
          "id": "CL:4052041"
        },
        {
          "id": "CL:4052042"
        },
        {
          "id": "CL:4052048"
        },
        {
          "id": "CL:4052049"
        },
        {
          "id": "CL:4052069"
        },
        {
          "id": "CL:4300031"
        },
        {
          "id": "CL:4307065"
        },
        {
          "id": "CL:7770004",
          "lbl": "suprabasal cell"
        },
        {
          "id": "FBbt:00000124"
        },
        {
          "id": "FBbt:00005038"
        },
        {
          "id": "FMA:14515"
        },
        {
          "id": "FMA:256165"
        },
        {
          "id": "FMA:66768"
        },
        {
          "id": "FMA:66778"
        },
        {
          "id": "FMA:67780"
        },
        {
          "id": "FMA:67978"
        },
        {
          "id": "FMA:69074"
        },
        {
          "id": "FMA:69075"
        },
        {
          "id": "FMA:69076"
        },
        {
          "id": "FMA:70552"
        },
        {
          "id": "FMA:70553"
        },
        {
          "id": "FMA:70555"
        },
        {
          "id": "FMA:70583"
        },
        {
          "id": "FMA:70605"
        },
        {
          "id": "FMA:70950"
        },
        {
          "id": "FMA:75802"
        },
        {
          "id": "KUPO:0001019"
        },
        {
          "id": "MESH:D005057"
        },
        {
          "id": "WBbt:0003672"
        },
        {
          "id": "WBbt:0005793"
        },
        {
          "id": "WBbt:0007846"
        },
        {
          "id": "Wikipedia:Syncytium"
        },
        {
          "id": "ZFA:0005323"
        },
        {
          "id": "ZFA:0009034"
        },
        {
          "id": "ZFA:0009035"
        },
        {
          "id": "ZFA:0009038"
        },
        {
          "id": "ZFA:0009039"
        },
        {
          "id": "ZFA:0009042"
        },
        {
          "id": "ZFA:0009050"
        },
        {
          "id": "ZFA:0009111"
        },
        {
          "id": "ZFA:0009148"
        },
        {
          "id": "ZFA:0009196"
        },
        {
          "id": "ZFA:0009372"
        },
        {
          "id": "ZFA:0009374"
        },
        {
          "id": "ZFA:0009383"
        },
        {
          "id": "ZFA:0009385"
        },
        {
          "id": "ZFA:0009388"
        }
      ],
      "edges": [
        {
          "sub": "CL:0000255",
          "pred": "owl:equivalentClass",
          "obj": "MESH:D005057",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0000414",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.9423076923076933"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "CALOHA:TS-2026",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7000000000000036"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "CARO:0000077",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7000000000000036"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "FBbt:00000124",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7000000000000036"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "FMA:66768",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7000000000000036"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "WBbt:0003672",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7000000000000036"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009034",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7000000000000036"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0000414",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.9423076923076933"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002077",
          "pred": "owl:equivalentClass",
          "obj": "FMA:69074",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002077",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009385",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000067",
          "pred": "owl:equivalentClass",
          "obj": "FMA:70605",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000067",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009035",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000068",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009372",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002078",
          "pred": "owl:equivalentClass",
          "obj": "FMA:69076",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002078",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009388",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000075",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009038",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000076",
          "pred": "owl:equivalentClass",
          "obj": "CALOHA:TS-1249",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000076",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009039",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000079",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009042",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002076",
          "pred": "owl:equivalentClass",
          "obj": "FMA:69075",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002076",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009383",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000098",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0004301",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000098",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009050",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000182",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0000575",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.9423076923076922"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000182",
          "pred": "owl:equivalentClass",
          "obj": "CALOHA:TS-0454",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000182",
          "pred": "owl:equivalentClass",
          "obj": "FMA:14515",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000182",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009111",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000182",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0000575",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.9423076923076922"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000244",
          "pred": "owl:equivalentClass",
          "obj": "FMA:66778",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000244",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009148",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000411",
          "pred": "owl:equivalentClass",
          "obj": "WBbt:0007846",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000411",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009196",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000418",
          "pred": "owl:equivalentClass",
          "obj": "WBbt:0005793",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000420",
          "pred": "owl:equivalentClass",
          "obj": "Wikipedia:Syncytium",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002204",
          "pred": "owl:equivalentClass",
          "obj": "FMA:67978",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002222",
          "pred": "owl:equivalentClass",
          "obj": "FMA:70950",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002327",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0004300",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002518",
          "pred": "owl:equivalentClass",
          "obj": "KUPO:0001019",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002518",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0009374",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002586",
          "pred": "owl:equivalentClass",
          "obj": "BTO:0004910",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0002586",
          "pred": "owl:equivalentClass",
          "obj": "FMA:75802",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0005006",
          "pred": "owl:equivalentClass",
          "obj": "ZFA:0005323",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0008026",
          "pred": "owl:equivalentClass",
          "obj": "FBbt:00005038",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:1000296",
          "pred": "owl:equivalentClass",
          "obj": "FMA:256165",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:1000415",
          "pred": "owl:equivalentClass",
          "obj": "FMA:67780",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:1000432",
          "pred": "owl:equivalentClass",
          "obj": "FMA:70552",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:1000433",
          "pred": "owl:equivalentClass",
          "obj": "FMA:70553",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:1000434",
          "pred": "owl:equivalentClass",
          "obj": "FMA:70555",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:1000441",
          "pred": "owl:equivalentClass",
          "obj": "FMA:70583",
          "meta": {
            "basicPropertyValues": [
              {
                "pred": "boomer:posteriorProbability",
                "val": "0.7"
              },
              {
                "pred": "boomer:priorProbability",
                "val": "0.7"
              }
            ]
          }
        },
        {
          "sub": "CL:0000018",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000019",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000021",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000026",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000032",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000034",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000059",
          "pred": "is_a",
          "obj": "CL:0002077"
        },
        {
          "sub": "CL:0000059",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000064",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000065",
          "pred": "is_a",
          "obj": "CL:0000067"
        },
        {
          "sub": "CL:0000066",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000066",
          "pred": "owl:disjointWith",
          "obj": "CL:0000738"
        },
        {
          "sub": "CL:0000067",
          "pred": "is_a",
          "obj": "CL:0000064"
        },
        {
          "sub": "CL:0000067",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000068",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000069",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:0000072",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:0000072",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0000075",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000076",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000077",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:0000077",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0000078",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:0000079",
          "pred": "is_a",
          "obj": "BFO:0000002"
        },
        {
          "sub": "CL:0000079",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000080",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000083",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0000098",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000098",
          "pred": "is_a",
          "obj": "CL:0000197"
        },
        {
          "sub": "CL:0000115",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000146",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0000148",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000150",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000152",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000160",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000163",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000164",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000165",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000168",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000179",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000182",
          "pred": "is_a",
          "obj": "BFO:0000002"
        },
        {
          "sub": "CL:0000182",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000182",
          "pred": "is_a",
          "obj": "CL:0000417"
        },
        {
          "sub": "CL:0000185",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0000187",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000188",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000197",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000209",
          "pred": "is_a",
          "obj": "CL:0000098"
        },
        {
          "sub": "CL:0000209",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0000214",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0000216",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000221",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000222",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000223",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000226",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000228",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000237",
          "pred": "is_a",
          "obj": "CL:0002077"
        },
        {
          "sub": "CL:0000239",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0000240",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:0000240",
          "pred": "is_a",
          "obj": "CL:0000079"
        },
        {
          "sub": "CL:0000241",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0000241",
          "pred": "is_a",
          "obj": "CL:0000079"
        },
        {
          "sub": "CL:0000242",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000244",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000255",
          "pred": "is_a",
          "obj": "CL:0000000"
        },
        {
          "sub": "CL:0000255",
          "pred": "owl:disjointWith",
          "obj": "CL:0000520"
        },
        {
          "sub": "CL:0000257",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000342",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000345",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000347",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000348",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000349",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000360",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000361",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000365",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000382",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000387",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000389",
          "pred": "is_a",
          "obj": "CL:0000658"
        },
        {
          "sub": "CL:0000407",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000411",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000411",
          "pred": "is_a",
          "obj": "CL:0000228"
        },
        {
          "sub": "CL:0000418",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000420",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000420",
          "pred": "is_a",
          "obj": "CL:4052002"
        },
        {
          "sub": "CL:0000425",
          "pred": "is_a",
          "obj": "CL:0000658"
        },
        {
          "sub": "CL:0000430",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000431",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000434",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000436",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000477",
          "pred": "is_a",
          "obj": "CL:0000500"
        },
        {
          "sub": "CL:0000487",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000500",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000517",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000518",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000521",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000571",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000574",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000575",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:0000618",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0000656",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000657",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000658",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000658",
          "pred": "is_a",
          "obj": "CL:0000327"
        },
        {
          "sub": "CL:0000659",
          "pred": "is_a",
          "obj": "CL:0000500"
        },
        {
          "sub": "CL:0000677",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000686",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0000710",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0000710",
          "pred": "is_a",
          "obj": "CL:0002077"
        },
        {
          "sub": "CL:0000722",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000730",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000731",
          "pred": "is_a",
          "obj": "CL:0000244"
        },
        {
          "sub": "CL:0000738",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000747",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000763",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000837",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000846",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0000851",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000852",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0000892",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0001035",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002031",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002032",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002062",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:0002075",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0002075",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:0002076",
          "pred": "is_a",
          "obj": "BFO:0000002"
        },
        {
          "sub": "CL:0002076",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002077",
          "pred": "is_a",
          "obj": "BFO:0000002"
        },
        {
          "sub": "CL:0002077",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002078",
          "pred": "is_a",
          "obj": "BFO:0000002"
        },
        {
          "sub": "CL:0002078",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002097",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0002149",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0002153",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002157",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0002159",
          "pred": "is_a",
          "obj": "CL:0002077"
        },
        {
          "sub": "CL:0002162",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002167",
          "pred": "is_a",
          "obj": "CL:0000098"
        },
        {
          "sub": "CL:0002174",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002179",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0002180",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0002181",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0002198",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002204",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002222",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002222",
          "pred": "is_a",
          "obj": "CL:0000306"
        },
        {
          "sub": "CL:0002224",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0002224",
          "pred": "is_a",
          "obj": "CL:0002222"
        },
        {
          "sub": "CL:0002231",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002236",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:0002237",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:0002239",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002249",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0002251",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002257",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002260",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002261",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002293",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002305",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0002306",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0002318",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002319",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002320",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002324",
          "pred": "is_a",
          "obj": "CL:0002327"
        },
        {
          "sub": "CL:0002326",
          "pred": "is_a",
          "obj": "CL:0002327"
        },
        {
          "sub": "CL:0002327",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002368",
          "pred": "is_a",
          "obj": "CL:0002076"
        },
        {
          "sub": "CL:0002491",
          "pred": "is_a",
          "obj": "CL:0000098"
        },
        {
          "sub": "CL:0002494",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002518",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002518",
          "pred": "is_a",
          "obj": "CL:1000497"
        },
        {
          "sub": "CL:0002519",
          "pred": "is_a",
          "obj": "CL:0002518"
        },
        {
          "sub": "CL:0002522",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002536",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002559",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0002565",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002565",
          "pred": "is_a",
          "obj": "CL:0000342"
        },
        {
          "sub": "CL:0002577",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002577",
          "pred": "is_a",
          "obj": "CL:0000349"
        },
        {
          "sub": "CL:0002584",
          "pred": "is_a",
          "obj": "CL:0002518"
        },
        {
          "sub": "CL:0002586",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0002586",
          "pred": "is_a",
          "obj": "CL:0000149"
        },
        {
          "sub": "CL:0002586",
          "pred": "is_a",
          "obj": "CL:0009004"
        },
        {
          "sub": "CL:0002625",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:0002665",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0005006",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0005009",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0005009",
          "pred": "is_a",
          "obj": "CL:0002518"
        },
        {
          "sub": "CL:0005010",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0005010",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:0005012",
          "pred": "is_a",
          "obj": "CL:0000067"
        },
        {
          "sub": "CL:0005012",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:0005013",
          "pred": "is_a",
          "obj": "CL:0000067"
        },
        {
          "sub": "CL:0007005",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0007021",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0007023",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0008026",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0008055",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:0009003",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0009005",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0009010",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0010002",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0010015",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:0011004",
          "pred": "is_a",
          "obj": "CL:0002222"
        },
        {
          "sub": "CL:0011026",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0011029",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0017000",
          "pred": "is_a",
          "obj": "CL:0005006"
        },
        {
          "sub": "CL:0017003",
          "pred": "is_a",
          "obj": "CL:1000296"
        },
        {
          "sub": "CL:0017010",
          "pred": "is_a",
          "obj": "CL:1000296"
        },
        {
          "sub": "CL:0017010",
          "pred": "is_a",
          "obj": "CL:4030024"
        },
        {
          "sub": "CL:0019001",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:0019026",
          "pred": "is_a",
          "obj": "CL:0000182"
        },
        {
          "sub": "CL:0019028",
          "pred": "is_a",
          "obj": "CL:0000182"
        },
        {
          "sub": "CL:0019029",
          "pred": "is_a",
          "obj": "CL:0000182"
        },
        {
          "sub": "CL:0019032",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:1000155",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:1000182",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:1000223",
          "pred": "is_a",
          "obj": "CL:0000098"
        },
        {
          "sub": "CL:1000272",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:1000296",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1000296",
          "pred": "is_a",
          "obj": "CL:1001320"
        },
        {
          "sub": "CL:1000415",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1000419",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:1000432",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1000433",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1000434",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1000441",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1000449",
          "pred": "is_a",
          "obj": "CL:0002518"
        },
        {
          "sub": "CL:1000497",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:1000600",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:1000601",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:1000703",
          "pred": "is_a",
          "obj": "CL:0002518"
        },
        {
          "sub": "CL:1000850",
          "pred": "is_a",
          "obj": "CL:0000067"
        },
        {
          "sub": "CL:1001430",
          "pred": "is_a",
          "obj": "CL:1000296"
        },
        {
          "sub": "CL:1001575",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:1001576",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:1001577",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:1001578",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:1001586",
          "pred": "is_a",
          "obj": "CL:0002327"
        },
        {
          "sub": "CL:1001590",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:1001590",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:1001591",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:1001592",
          "pred": "is_a",
          "obj": "CL:1000415"
        },
        {
          "sub": "CL:1001597",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:1001598",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:1100001",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:1100001",
          "pred": "is_a",
          "obj": "CL:0000151"
        },
        {
          "sub": "CL:2000020",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:2000021",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:2000064",
          "pred": "is_a",
          "obj": "CL:0002078"
        },
        {
          "sub": "CL:2000084",
          "pred": "is_a",
          "obj": "CL:1000432"
        },
        {
          "sub": "CL:2000094",
          "pred": "is_a",
          "obj": "CL:1000441"
        },
        {
          "sub": "CL:4030007",
          "pred": "is_a",
          "obj": "CL:4052018"
        },
        {
          "sub": "CL:4030012",
          "pred": "is_a",
          "obj": "CL:0000076"
        },
        {
          "sub": "CL:4030023",
          "pred": "is_a",
          "obj": "CL:4030024"
        },
        {
          "sub": "CL:4030024",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:4030031",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:4030066",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:4030066",
          "pred": "is_a",
          "obj": "CL:0002518"
        },
        {
          "sub": "CL:4032000",
          "pred": "is_a",
          "obj": "CL:1000296"
        },
        {
          "sub": "CL:4033013",
          "pred": "is_a",
          "obj": "CL:7770004"
        },
        {
          "sub": "CL:4033023",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:4033048",
          "pred": "is_a",
          "obj": "CL:7770004"
        },
        {
          "sub": "CL:4033055",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:4033066",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:4040006",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:4042019",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:4047056",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:4052018",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "CL:4052019",
          "pred": "is_a",
          "obj": "CL:4052018"
        },
        {
          "sub": "CL:4052033",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052034",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052035",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:4052035",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052036",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052039",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052040",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052041",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052042",
          "pred": "is_a",
          "obj": "CL:0002204"
        },
        {
          "sub": "CL:4052042",
          "pred": "is_a",
          "obj": "CL:1000296"
        },
        {
          "sub": "CL:4052048",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:4052049",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:4052049",
          "pred": "is_a",
          "obj": "CL:0000075"
        },
        {
          "sub": "CL:4052069",
          "pred": "is_a",
          "obj": "CL:0000068"
        },
        {
          "sub": "CL:4300031",
          "pred": "is_a",
          "obj": "CL:0000255"
        },
        {
          "sub": "CL:4307065",
          "pred": "is_a",
          "obj": "CL:1100001"
        },
        {
          "sub": "CL:7770004",
          "pred": "is_a",
          "obj": "CL:0000066"
        },
        {
          "sub": "BTO:0001514",
          "pred": "is_a",
          "obj": "BTO:0000414"
        },
        {
          "sub": "BTO:0001540",
          "pred": "is_a",
          "obj": "BTO:0000414"
        },
        {
          "sub": "BTO:0001663",
          "pred": "is_a",
          "obj": "BTO:0000414"
        },
        {
          "sub": "BTO:0002295",
          "pred": "is_a",
          "obj": "BTO:0000414"
        },
        {
          "sub": "BTO:0002309",
          "pred": "is_a",
          "obj": "BTO:0000414"
        },
        {
          "sub": "BTO:0002484",
          "pred": "is_a",
          "obj": "BTO:0000414"
        },
        {
          "sub": "BTO:0003736",
          "pred": "is_a",
          "obj": "BTO:0000414"
        }
      ]
    }
  ]
}```

## Summary

This tutorial demonstrated the full BOOMER pipeline on real ontologies:

```bash
# Download
curl -sL -o cl.obo http://purl.obolibrary.org/obo/cl.obo
curl -sL -o bto.obo http://purl.obolibrary.org/obo/bto.obo

# Convert & merge (via Python API for label matching)
# ... see cells above ...

# Extract neighborhood
pyboomer extract merged.yaml --id CL:0000066 --max-hops 1 -o cluster.yaml

# Solve
pyboomer solve cluster.yaml --timeout 120 -C 20 -O yaml -o solution.yaml

# Export
pyboomer solve cluster.yaml --timeout 120 -C 20 -O sssom -o solution.sssom.tsv
pyboomer solve cluster.yaml --timeout 120 -C 20 -O obographs -o solution.obographs.json
```

**Key takeaways:**

- Label matching is a simple heuristic; real pipelines use curated SSSOM files
  or tools like [OAK](https://incatools.github.io/ontology-access-kit/).
- `MemberOfDisjointGroup` constraints (auto-generated by prefix) help BOOMER
  resolve ambiguity when a CL term could match multiple BTO terms.
- Module extraction (`--max-hops`) is essential for tractability on large ontologies.
- The `-C` (max pfacts per clique) flag controls the partition granularity,
  trading off between speed and joint reasoning across related facts.
- SSSOM and OBOGraphs exports make results interoperable with the broader
  OBO ecosystem.